In [4]:
from pathlib import Path; print('\n'.join(sorted(f.name for f in Path(r"C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun").iterdir())))







.ipynb_checkpoints
Alkalinity_zonal.xlsx
Copy of R-80 zonal data.xlsx
EDDtoIntellusNamesV2.xlsx
EIM_07_15_2026.csv
R-80 data rotated.csv
WellsOfInterest.xls
WellsOfInterest_v2.xls
~$Copy of R-80 zonal data.xlsx


In [15]:
# Cell 2 — Process zonal data into review tables for zonal screening and downstream PCA work
#
# Outputs:
# 1. zonal_detects_matrix.xlsx
#    Sample-by-analyte categorical screening matrix.
#    Zones are ordered 7, 6, 5, 4, 3, 2, 1 from top to bottom.
#
# 2. zonal_rotated_numeric_matrix.xlsx
#    Sample-by-analyte numeric matrix for downstream multivariate work.
#
# 3. zonal_value_qualifier_pql_matrix.xlsx
#    Review matrix with MultiIndex columns:
#        analyte -> {value, qualifier, pql}
#    The qualifier field in this table is the resolved output qualifier:
#    either the original lab qualifier or our elaborated code such as D or D_sub.
#
# Key logic notes:
# - ALK is preferred when directly reported.
# - If ALK is absent, ALK-HCO3 is remapped into ALK for output purposes.
# - Alternate alkalinity forms are removed from final output tables after remap.
# - LAB_DETECTION_LIMIT is treated as the PQL.
# - A row earns D (or D_sub for substituted ALK) only when LAB_RESULT > PQL.
# - Otherwise, the detect/category code comes from LAB_QUALIFIER.

from pathlib import Path
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# User settings
# ------------------------------------------------------------------
WORK_DIR = Path(r"C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun")
INPUT_FILE = WORK_DIR / "copy of R-80 zonal data.xlsx"
INPUT_SHEET = 0

RESULTS_DIR = WORK_DIR / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUT_DETECTS = RESULTS_DIR / "zonal_detects_matrix.xlsx"
OUT_ROTATED = RESULTS_DIR / "zonal_rotated_numeric_matrix.xlsx"
OUT_VALUE_QUAL = RESULTS_DIR / "zonal_value_qualifier_pql_matrix.xlsx"

# Use ALK-HCO3 as fallback ALK where ALK is missing.
ALK_REMAP_SOURCE = "ALK-HCO3"

# Remove alternate alkalinity forms from final outputs after remapping.
DROP_ANALYTES = {
    "ALK-CO3",
    "ALK-CO3+HCO3",
    "ALK-HCO3",
    "ALK-OH",
    "ALK-PHEN",
    "CO3(-2)",
    "HCO3(-1)",
}

# ------------------------------------------------------------------
# Read and validate input
# ------------------------------------------------------------------
raw = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET, dtype=str)
raw.columns = [str(c).strip() for c in raw.columns]

required_cols = [
    "FIELD_SAMPLE_ID",
    "PARAMETER_CODE",
    "PARAMETER_NAME",
    "LAB_RESULT",
    "LAB_QUALIFIER",
    "ANALYSIS_DATE",
    "LAB_DETECTION_LIMIT",
]
missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns in zonal file: {missing}")

df = raw.copy()

sample = df["FIELD_SAMPLE_ID"].fillna("").astype(str).str.strip()
analyte = df["PARAMETER_CODE"].fillna("").astype(str).str.strip().str.upper()
analyte_name = df["PARAMETER_NAME"].fillna("").astype(str).str.strip()
qual = df["LAB_QUALIFIER"].fillna("").astype(str).str.strip().str.upper()
res_txt = df["LAB_RESULT"].fillna("").astype(str).str.strip()
pql_txt = df["LAB_DETECTION_LIMIT"].fillna("").astype(str).str.strip()

# ------------------------------------------------------------------
# Parse numeric fields
# Numeric parsing extracts the first numeric token after stripping
# leading comparator symbols and commas.
# ------------------------------------------------------------------
res_core = (
    res_txt
    .str.replace(r"^[<>]\s*", "", regex=True)
    .str.replace(",", "", regex=False)
)
pql_core = (
    pql_txt
    .str.replace(r"^[<>]\s*", "", regex=True)
    .str.replace(",", "", regex=False)
)

res_num = pd.to_numeric(
    res_core.str.extract(r"([-+]?\d*\.?\d+)")[0],
    errors="coerce"
)
pql_num = pd.to_numeric(
    pql_core.str.extract(r"([-+]?\d*\.?\d+)")[0],
    errors="coerce"
)

# ------------------------------------------------------------------
# Build base detect/category state
#
# Rule:
# - D is assigned only when result_num > pql_num.
# - Otherwise use LAB_QUALIFIER if present.
# - If qualifier is blank and result is not above PQL, fall back to:
#     N  for explicit nondetect-style reporting
#     NF for anything else not clearly classifiable
# ------------------------------------------------------------------
nd_text = res_txt.str.contains(
    r"\bND\b|NON[- ]?DETECT|NOT DETECT|BDL",
    case=False,
    regex=True,
    na=False,
)
above_pql = res_num.notna() & pql_num.notna() & (res_num > pql_num)

state = np.where(
    above_pql,
    "D",
    np.where(
        qual != "",
        qual,
        np.where(nd_text, "N", "NF")
    )
)

# ------------------------------------------------------------------
# Build working long table
# ------------------------------------------------------------------
zonal_long = pd.DataFrame({
    "sample": sample,
    "analyte": analyte,
    "analyte_name": analyte_name,
    "result_text": res_txt,
    "result_num": res_num,
    "qualifier": qual,
    "pql": pql_num,
    "state": state,
    "analysis_date": pd.to_datetime(df["ANALYSIS_DATE"], errors="coerce"),
})

zonal_long = zonal_long[
    (zonal_long["sample"] != "") &
    (zonal_long["analyte"] != "")
].copy()

# ------------------------------------------------------------------
# Derive zone number and order samples from Zone 7 to Zone 1
# Expected naming pattern: Zone 1, Zone 2, ... Zone 7
# ------------------------------------------------------------------
def extract_zone_number(text):
    if pd.isna(text):
        return np.nan
    m = re.search(r"\bZONE\s*([0-9]+)\b", str(text).upper())
    return int(m.group(1)) if m else np.nan

zonal_long["zone"] = zonal_long["sample"].apply(extract_zone_number)

sample_zone = zonal_long[["sample", "zone"]].drop_duplicates().copy()
sample_order = (
    sample_zone
    .sort_values(["zone", "sample"], ascending=[False, True], na_position="last")
    ["sample"]
    .tolist()
)

# ------------------------------------------------------------------
# Keep the best row per sample/analyte.
# Preference order:
# 1. higher information rank
# 2. higher numeric value
# 3. more recent analysis date
# ------------------------------------------------------------------
info_rank_map = {
    "D": 6,
    "D_SUB": 5,
    "J": 4,
    "UJ": 3,
    "N": 2,
    "NF": 1,
}
zonal_long["info_rank"] = zonal_long["state"].map(info_rank_map).fillna(2).astype(int)

best_rows = (
    zonal_long
    .sort_values(
        ["sample", "analyte", "info_rank", "result_num", "analysis_date"],
        ascending=[True, True, False, False, False],
        na_position="last"
    )
    .drop_duplicates(["sample", "analyte"], keep="first")
    .copy()
)

# ------------------------------------------------------------------
# Remap ALK-HCO3 into ALK when ALK is absent.
# The substituted rows retain the same value, qualifier, and pql.
# Their detect/category code becomes D_sub only if result > pql.
# Otherwise the detect/category code remains the LAB_QUALIFIER-derived code.
# ------------------------------------------------------------------
best_rows["alk_substituted"] = False

alk_rows = best_rows[best_rows["analyte"] == "ALK"].copy()
alt_alk_rows = best_rows[best_rows["analyte"] == ALK_REMAP_SOURCE].copy()

samples_with_alk = set(alk_rows["sample"])
alt_alk_sub = alt_alk_rows[~alt_alk_rows["sample"].isin(samples_with_alk)].copy()

if not alt_alk_sub.empty:
    alt_alk_sub["analyte"] = "ALK"
    alt_alk_sub["alk_substituted"] = True
    alt_alk_sub["state"] = np.where(
        alt_alk_sub["result_num"].notna() &
        alt_alk_sub["pql"].notna() &
        (alt_alk_sub["result_num"] > alt_alk_sub["pql"]),
        "D_sub",
        np.where(
            alt_alk_sub["qualifier"] != "",
            alt_alk_sub["qualifier"],
            np.where(
                alt_alk_sub["result_text"].str.contains(
                    r"\bND\b|NON[- ]?DETECT|NOT DETECT|BDL",
                    case=False,
                    regex=True,
                    na=False,
                ),
                "N",
                "NF"
            )
        )
    )

final_rows = pd.concat([best_rows, alt_alk_sub], ignore_index=True)

# Remove alternate alkalinity analytes from final outputs after remap.
final_rows = final_rows[
    ~final_rows["analyte"].isin(DROP_ANALYTES)
].copy()

# If both direct ALK and substituted ALK exist, keep direct ALK.
final_rows["alk_priority"] = np.where(final_rows["alk_substituted"], 0, 1)

final_rows = (
    final_rows
    .sort_values(
        ["sample", "analyte", "alk_priority", "info_rank", "result_num", "analysis_date"],
        ascending=[True, True, False, False, False, False],
        na_position="last"
    )
    .drop_duplicates(["sample", "analyte"], keep="first")
    .copy()
)

# Refresh information rank after ALK substitution state updates.
final_rows["info_rank"] = final_rows["state"].map(info_rank_map).fillna(2).astype(int)

# ------------------------------------------------------------------
# Table 1 — Detects matrix
# This is the categorical screening matrix used for quick review.
# ------------------------------------------------------------------
detect_priority = {
    "D_sub": 7,
    "D": 6,
    "J": 5,
    "UJ": 4,
    "N": 3,
    "NF": 2,
}

detects_long = final_rows[["sample", "analyte", "state"]].copy()
detects_long["prio"] = detects_long["state"].map(detect_priority).fillna(1).astype(int)

detects_long = (
    detects_long
    .sort_values(["sample", "analyte", "prio"], ascending=[True, True, False])
    .drop_duplicates(["sample", "analyte"], keep="first")
)

detects_matrix = (
    detects_long
    .pivot(index="sample", columns="analyte", values="state")
    .fillna("NF")
)
detects_matrix = detects_matrix.reindex(sample_order).sort_index(axis=1)

# ------------------------------------------------------------------
# Table 2 — Rotated numeric matrix
# Only direct-above-PQL detections and substituted-above-PQL detections
# carry numeric values into this matrix.
# ------------------------------------------------------------------
rot_long = final_rows[["sample", "analyte", "state", "result_num"]].copy()
rot_long["value_num"] = np.where(
    rot_long["state"].isin(["D", "D_sub"]),
    rot_long["result_num"],
    np.nan
)

rotated_matrix = rot_long.pivot(index="sample", columns="analyte", values="value_num")
rotated_matrix = rotated_matrix.reindex(sample_order).sort_index(axis=1)

# ------------------------------------------------------------------
# Table 3 — Value / qualifier / pql matrix
# MultiIndex column structure:
#   analyte -> {value, qualifier, pql}
# The qualifier field here is the resolved output qualifier/category:
# it may be the original lab qualifier or our elaborated code (e.g., D, D_sub).
# ------------------------------------------------------------------
vq_long = final_rows[["sample", "analyte", "result_num", "state", "pql"]].copy()
vq_long = vq_long.rename(columns={
    "result_num": "value",
    "state": "qualifier",
})

value_part = vq_long.pivot(index="sample", columns="analyte", values="value")
qual_part = vq_long.pivot(index="sample", columns="analyte", values="qualifier")
pql_part = vq_long.pivot(index="sample", columns="analyte", values="pql")

value_part.columns = pd.MultiIndex.from_product([value_part.columns, ["value"]])
qual_part.columns = pd.MultiIndex.from_product([qual_part.columns, ["qualifier"]])
pql_part.columns = pd.MultiIndex.from_product([pql_part.columns, ["pql"]])

value_qualifier_matrix = pd.concat([value_part, qual_part, pql_part], axis=1)

ordered_analytes = sorted(set(value_qualifier_matrix.columns.get_level_values(0)))
ordered_cols = []
for a in ordered_analytes:
    for sub in ["value", "qualifier", "pql"]:
        if (a, sub) in value_qualifier_matrix.columns:
            ordered_cols.append((a, sub))

value_qualifier_matrix = value_qualifier_matrix.reindex(
    columns=pd.MultiIndex.from_tuples(ordered_cols)
)
value_qualifier_matrix = value_qualifier_matrix.reindex(sample_order)

# ------------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------------
detects_matrix.to_excel(OUT_DETECTS)
rotated_matrix.to_excel(OUT_ROTATED)
value_qualifier_matrix.to_excel(OUT_VALUE_QUAL)

print("\nCell 2 complete. Files written:")
for f in [OUT_DETECTS, OUT_ROTATED, OUT_VALUE_QUAL]:
    print(f" - {f}")


Cell 2 complete. Files written:
 - C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\Results\zonal_detects_matrix.xlsx
 - C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\Results\zonal_rotated_numeric_matrix.xlsx
 - C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\Results\zonal_value_qualifier_pql_matrix.xlsx


In [18]:
import pandas as pd
from pathlib import Path

work_dir = Path(r"C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun")

# 1) EIM file
eim_file = work_dir / "EIM_EXPORT_08_25_2026.csv"
eim = pd.read_csv(eim_file, nrows=5)
print("EIM columns:", eim.columns.tolist())

# 2) Wells of interest
woi = pd.read_excel(work_dir / "WellsOfInterest_v2.xls", nrows=5)
print("WOI columns:", woi.columns.tolist())

# 3) EDD mapping
map_file = work_dir / "EDDtoIntellusNamesV2.xlsx"
mapping = pd.read_excel(map_file, nrows=5)
print("Mapping columns:", mapping.columns.tolist())

# 4) Zonal matrix columns
zonal = pd.read_excel(work_dir / "Results" / "zonal_value_qualifier_pql_matrix.xlsx", header=[0,1], nrows=2)
print("Zonal top-level analytes:", zonal.columns.get_level_values(0).unique().tolist()[:20])


EIM columns: ['Site ID', 'Screening Flag', 'Location', 'Field Sample ID', 'Date Sampled', 'Parameter', 'Result', 'Units', 'Lab Qualifier', 'Validation Qualifier', 'Detect?', 'Matrix', 'Purpose', 'Type', 'Time', 'Program', 'Event', 'Analysis Type', 'Filtered', 'Leached', 'Result Type Code', 'Analytical Method', 'Start Depth', 'End Depth', 'Depth Units', 'Excavated Flag', 'Use Flag']
WOI columns: ['OBJECTID', 'Site_ID', 'Location_ID', 'Location_Type', 'Well_ID', 'Ground_Elevation', 'Well_Total_Depth', 'Depth_Units', 'Well_Diameter', 'Well_Diameter_Units', 'Well_Use', 'Well_Status', 'Well_Installation_Date', 'Drilling_Method_Code', 'Perforation_Zone_Start_Depth', 'Perforation_Zone_End_Depth', 'Geological_Unit_Code', 'Aquifer', 'Hydrostratigraphic_Unit', 'Chamber_Effective_Top_Depth', 'Chamber_Effective_Bottom_Depth', 'Well_Completion_Report_Url', 'Latitude', 'Longitude', 'Top_of_Casing_Elevation', 'Cat']
Mapping columns: ['EDD', 'Intellus']
Zonal top-level analytes: ['Unnamed: 0_level_0',

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

RUN_DIR = Path(r"C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun")
EIM_PATH = RUN_DIR / "EIM_07_15_2026.csv"
raw = pd.read_csv(EIM_PATH, dtype=str, low_memory=False)
raw.columns = [c.strip() for c in raw.columns]

# ---------- map required columns ----------
def norm(s): 
    return re.sub(r'[^a-z0-9]+', '_', str(s).lower()).strip('_')

nmap = {c: norm(c) for c in raw.columns}

def find_one(patterns):
    rx = re.compile("|".join(patterns), re.I)
    for c in raw.columns:
        if rx.search(c) or rx.search(nmap[c]):
            return c
    return None

sample_col = find_one([r'^sample(_id)?$', r'sample.*id', r'field.*id', r'location.*id', r'station.*id', r'^site$'])
analyte_col = find_one([r'^analyte$', r'constituent', r'parameter', r'chemical'])
detect_col = find_one([r'^detect$', r'detected', r'detect_flag'])
valqual_col = find_one([r'validation_qualifier', r'validation.*qual', r'validator_qual', r'val.*qual'])

print("sample_col :", sample_col)
print("analyte_col:", analyte_col)
print("detect_col :", detect_col)
print("valqual_col:", valqual_col)

missing = [k for k,v in {
    "sample_col": sample_col,
    "analyte_col": analyte_col,
    "detect_col": detect_col,
    "valqual_col": valqual_col
}.items() if v is None]
if missing:
    raise ValueError(f"Could not auto-map required columns: {missing}\n"
                     f"Available columns:\n{list(raw.columns)}")

df = raw.copy()
sample = df[sample_col].fillna("").astype(str).str.strip()
analyte = df[analyte_col].fillna("").astype(str).str.strip()
detect = df[detect_col].fillna("").astype(str).str.strip().str.upper()
vqual = df[valqual_col].fillna("").astype(str).str.upper()

# ---------- D/N/J/NF logic ----------
# Rule:
# - Detect == Y and Validation Qualifier contains J -> J
# - Detect == Y and no J -> D
# - Detect in N/NO/FALSE/0 -> N
# - Otherwise -> NF
is_y = detect.isin(["Y","YES","TRUE","T","1"])
is_n = detect.isin(["N","NO","FALSE","F","0"])
has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)

state = np.where(
    is_y & has_j, "J",
    np.where(
        is_y & ~has_j, "D",
        np.where(is_n, "N", "NF")
    )
)

work = pd.DataFrame({
    "sample": sample,
    "analyte": analyte,
    "state": state
})

# If sample/analyte missing, force NF
work.loc[(work["sample"] == "") | (work["analyte"] == ""), "state"] = "NF"

# Resolve duplicates per sample+analyte with precedence J > D > N > NF
priority = {"J": 4, "D": 3, "N": 2, "NF": 1}
work["prio"] = work["state"].map(priority).fillna(0).astype(int)
work = (work.sort_values("prio", ascending=False)
            .drop_duplicates(subset=["sample","analyte"], keep="first"))

ternary = work.pivot(index="sample", columns="analyte", values="state").fillna("NF")

# Also remove blank sample/analyte axes if present
ternary = ternary.loc[ternary.index != "", ternary.columns != ""]

out_path = RUN_DIR / "EIM_07_15_2026__ternary_DNJNF_matrix.csv"
ternary.to_csv(out_path)

print("Saved:", out_path)
print("Shape:", ternary.shape)
display(ternary.head(10))

# Quick counts
counts = ternary.stack().value_counts(dropna=False)
print("\nState counts:")
display(counts.to_frame("count"))

sample_col : Field Sample ID
analyte_col: Parameter
detect_col : Detect?
valqual_col: Validation Qualifier


C:\Users\kylian.robinson\AppData\Local\Temp\ipykernel_9540\3434267374.py:58: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)


Saved: C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\EIM_07_15_2026__ternary_DNJNF_matrix.csv
Shape: (5801, 434)


analyte,"1,3,5-Naphthalene trisulfonic acid","1,3,6-Naphthalene trisulfonic acid","1,5-Naphthalenedisulfonic acid","1,6-Naphthalene disulfonic acid",1-Naphthalene sulfonic acid,11-Chloroeicosafluoro-3-oxaundecane-1-sulfonic acid,"1H, 1H, 2H, 2H-Perfluorododecanesulphonic acid","1H, 1H, 2H, 2H-Perfluorohexanesulfonic acid","1H, 1H, 2H, 2H-perfluorodecane sulfonic acid","1H, 1H, 2H, 2H-perfluorooctane sulfonic acid",...,Uranium-238,Vanadium,Vinyl Chloride,Vinyl acetate,Xylene (Total),"Xylene[1,2-]","Xylene[1,3-]+Xylene[1,4-]",Zinc,"cis-1,4-Dichloro-2-butene",n-Heptane
sample,,,,,,,,,,,,,,,,,,,,,
CALA-24-325466,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,D,NF,N,N,NF,N,N,NF,NF,NF
CALA-24-325467,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,D,NF,NF,NF,NF,NF,J,NF,NF
CALA-24-325468,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,N,N,NF,N,N,NF,NF,NF
CALA-24-331364,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
CALA-24-331375,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
CALA-24-331376,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
CALA-24-331377,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
CALA-24-331378,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
CALA-24-333782,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF



State counts:


,count
NF,2319304
N,151699
D,33950
J,12681


In [4]:
import re
import numpy as np
import pandas as pd

# expects `raw` already loaded from EIM csv in prior cells
df = raw.copy()
df.columns = [c.strip() for c in df.columns]

def norm(s): 
    return re.sub(r'[^a-z0-9]+', '_', str(s).lower()).strip('_')

nmap = {c: norm(c) for c in df.columns}

def find_one(patterns):
    rx = re.compile("|".join(patterns), re.I)
    for c in df.columns:
        if rx.search(c) or rx.search(nmap[c]):
            return c
    return None

# ---- map columns (adjust manually if auto-map misses) ----
sample_col   = find_one([r'^sample(_id)?$', r'sample.*id', r'field.*id', r'location.*id', r'station.*id', r'^site$'])
analyte_col  = find_one([r'^analyte$', r'constituent', r'parameter', r'chemical'])
result_col   = find_one([r'final_result', r'reported_result', r'(^|_)result($|_)', r'conc', r'value'])
detect_col   = find_one([r'^detect$', r'detected', r'detect_flag'])
valqual_col  = find_one([r'validation_qualifier', r'validation.*qual', r'validator_qual', r'val.*qual'])
year_col     = find_one([r'^year$', r'sample_date', r'collection_date', r'collected', r'analysis_date', r'date'])

print("Mapped columns:")
print(" analyte :", analyte_col)
print(" result  :", result_col)
print(" detect  :", detect_col)
print(" valqual :", valqual_col)
print(" year src:", year_col)

req = [analyte_col, result_col, detect_col, valqual_col, year_col]
if any(c is None for c in req):
    missing = [name for name, c in zip(["analyte","result","detect","valqual","year"], req) if c is None]
    raise ValueError(f"Missing required columns for this method: {missing}")

# ---- parse year ----
yr = pd.to_datetime(df[year_col], errors="coerce").dt.year
# if year column is already numeric-ish text, backfill:
yr2 = pd.to_numeric(df[year_col], errors="coerce")
yr = yr.fillna(yr2).astype("Int64")

# ---- parse numeric result ----
res_txt = df[result_col].fillna("").astype(str).str.strip()
res_core = res_txt.str.replace(r'^[<>]\s*', '', regex=True).str.replace(",", "", regex=False)
res_num = pd.to_numeric(res_core, errors="coerce")

detect = df[detect_col].fillna("").astype(str).str.strip().str.upper()
vqual  = df[valqual_col].fillna("").astype(str).str.upper()
analyte = df[analyte_col].fillna("").astype(str).str.strip()

is_detect_y = detect.isin(["Y","YES","TRUE","T","1"])
has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)

# J-detect and NQ-detect definitions
is_j_detect  = is_detect_y & has_j & res_num.notna()
is_nq_detect = is_detect_y & (~has_j) & res_num.notna()

work = pd.DataFrame({
    "year": yr,
    "analyte": analyte,
    "result_num": res_num,
    "is_j_detect": is_j_detect,
    "is_nq_detect": is_nq_detect
})

# keep valid keys only
work = work[(work["year"].notna()) & (work["analyte"] != "")]

# ---- aggregate by year/analyte ----
agg = (work.groupby(["year","analyte"], dropna=False)
          .apply(lambda g: pd.Series({
              "highest_J": g.loc[g["is_j_detect"], "result_num"].max() if g["is_j_detect"].any() else np.nan,
              "lowest_NQ": g.loc[g["is_nq_detect"], "result_num"].min() if g["is_nq_detect"].any() else np.nan,
              "n_J": int(g["is_j_detect"].sum()),
              "n_NQ": int(g["is_nq_detect"].sum())
          }))
          .reset_index())

# PQL test: plausible bracket exists when both values exist and highest_J <= lowest_NQ
agg["pql_bracket_exists"] = (
    agg["highest_J"].notna() &
    agg["lowest_NQ"].notna() &
    (agg["highest_J"] <= agg["lowest_NQ"])
)

# optional midpoint estimate when bracket exists
agg["pql_mid_est"] = np.where(
    agg["pql_bracket_exists"],
    (agg["highest_J"] + agg["lowest_NQ"]) / 2.0,
    np.nan
)

# ---- output table with two subcolumns per analyte ----
wide = agg.pivot(index="year", columns="analyte", values=["highest_J","lowest_NQ"])
wide = wide.sort_index(axis=0).sort_index(axis=1)

# companion diagnostics (same shape idea for test flags if needed)
wide_test = agg.pivot(index="year", columns="analyte", values="pql_bracket_exists")
wide_mid  = agg.pivot(index="year", columns="analyte", values="pql_mid_est")

# ---- save ----
out_base = RUN_DIR / "EIM_07_15_2026__PQL_bracket_by_year_analyte"
wide.to_csv(str(out_base) + "__highJ_lowNQ.csv")
wide_test.to_csv(str(out_base) + "__bracket_test.csv")
wide_mid.to_csv(str(out_base) + "__mid_estimate.csv")
agg.to_csv(str(out_base) + "__long_diagnostics.csv", index=False)

print("Saved:")
print(" ", str(out_base) + "__highJ_lowNQ.csv")
print(" ", str(out_base) + "__bracket_test.csv")
print(" ", str(out_base) + "__mid_estimate.csv")
print(" ", str(out_base) + "__long_diagnostics.csv")

display(wide.head(10))
display(agg.head(20))

Mapped columns:
 analyte : Parameter
 result  : Result
 detect  : Detect?
 valqual : Validation Qualifier
 year src: Date Sampled


C:\Users\kylian.robinson\AppData\Local\Temp\ipykernel_15468\1369755024.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)


Saved:
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\EIM_07_15_2026__PQL_bracket_by_year_analyte__highJ_lowNQ.csv
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\EIM_07_15_2026__PQL_bracket_by_year_analyte__bracket_test.csv
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\EIM_07_15_2026__PQL_bracket_by_year_analyte__mid_estimate.csv
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\EIM_07_15_2026__PQL_bracket_by_year_analyte__long_diagnostics.csv


highest_J                                     \
analyte 1,3,5-Naphthalene trisulfonic acid 1,3,6-Naphthalene trisulfonic acid   
year                                                                            
2024                               0.00734                            0.00949   
2025                               0.04900                            0.02740   
2026                                   NaN                            0.00432   

                                                                        \
analyte 1,5-Naphthalenedisulfonic acid 1,6-Naphthalene disulfonic acid   
year                                                                     
2024                           0.00424                             NaN   
2025                           0.00950                             NaN   
2026                           0.01330                             NaN   

                                     \
analyte 1-Naphthalene sulfonic acid   
year                                  
2024                            NaN   
2025                            NaN   
2026                            NaN   

                                                             \
analyte 11-Chloroeicosafluoro-3-oxaundecane-1-sulfonic acid   
year                                                          
2024                                                   NaN    
2025                                                   NaN    
2026                                                   NaN    

                                                        \
analyte 1H, 1H, 2H, 2H-Perfluorododecanesulphonic acid   
year                                                     
2024                                               NaN   
2025                                               NaN   
2026                                               NaN   

                                                     \
analyte 1H, 1H, 2H, 2H-Perfluorohexanesulfonic acid   
year                                                  
2024                                            NaN   
2025                                            NaN   
2026                                            NaN   

                                                      \
analyte 1H, 1H, 2H, 2H-perfluorodecane sulfonic acid   
year                                                   
2024                                             NaN   
2025                                             NaN   
2026                                             NaN   

                                                      ...   lowest_NQ  \
analyte 1H, 1H, 2H, 2H-perfluorooctane sulfonic acid  ... Uranium-238   
year                                                  ...               
2024                                           13.10  ...      0.0789   
2025                                            3.85  ...      0.0258   
2026                                             NaN  ...      0.0693   

                                                                           \
analyte Vanadium Vinyl Chloride Vinyl acetate Xylene (Total) Xylene[1,2-]   
year                                                                        
2024     4.63000           28.4           NaN            NaN        246.0   
2025     3.72596            NaN           NaN            NaN         63.4   
2026     3.72000            NaN           NaN            NaN          NaN   

                                                                              
analyte Xylene[1,3-]+Xylene[1,4-]   Zinc cis-1,4-Dichloro-2-butene n-Heptane  
year                                                                          
2024                         52.5  20.20                       NaN      97.1  
2025                         79.0   4.25                       NaN       NaN  
2026                          NaN  57.60                       NaN       NaN  

[3 rows x 868 columns]

,year,analyte,highest_J,lowest_NQ,n_J,n_NQ,pql_bracket_exists,pql_mid_est
0,2024,"1,3,5-Naphthalene trisulfonic acid",0.00734,0.0172,4.0,5.0,True,0.012270
1,2024,"1,3,6-Naphthalene trisulfonic acid",0.00949,0.0143,2.0,10.0,True,0.011895
2,2024,"1,5-Naphthalenedisulfonic acid",0.00424,NaN,2.0,0.0,False,NaN
3,2024,"1,6-Naphthalene disulfonic acid",NaN,NaN,0.0,0.0,False,NaN
4,2024,1-Naphthalene sulfonic acid,NaN,NaN,0.0,0.0,False,NaN
5,2024,11-Chloroeicosafluoro-3-oxaundecane-1-sulfonic...,NaN,NaN,0.0,0.0,False,NaN
6,2024,"1H, 1H, 2H, 2H-Perfluorododecanesulphonic acid",NaN,NaN,0.0,0.0,False,NaN
7,2024,"1H, 1H, 2H, 2H-Perfluorohexanesulfonic acid",NaN,NaN,0.0,0.0,False,NaN
8,2024,"1H, 1H, 2H, 2H-perfluorodecane sulfonic acid",NaN,NaN,0.0,0.0,False,NaN
9,2024,"1H, 1H, 2H, 2H-perfluorooctane sulfonic acid",13.10000,NaN,1.0,0.0,False,NaN


In [7]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

RUN_DIR = Path(r"C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun")
IN_PATH = RUN_DIR / "copy of R-80 zonal data.xlsx"
raw = pd.read_excel(IN_PATH, dtype=str)
raw.columns = [str(c).strip() for c in raw.columns]

# Explicit mapping for this file
sample_col  = "FIELD_SAMPLE_ID"
analyte_col = "PARAMETER_NAME"
result_col  = "LAB_RESULT"
valqual_col = "LAB_QUALIFIER"
date_col    = "SAMPLE_DATE"
pql_col     = "LAB_DETECTION_LIMIT"          # use as PQL proxy in this block
mdl_col     = "METHOD_DETECTION_LIMIT"

for c in [sample_col, analyte_col, result_col, valqual_col, date_col]:
    if c not in raw.columns:
        raise ValueError(f"Missing expected column: {c}")

df = raw.copy()

sample = df[sample_col].fillna("").astype(str).str.strip()
analyte = df[analyte_col].fillna("").astype(str).str.strip()
vqual = df[valqual_col].fillna("").astype(str).str.upper()

res_txt = df[result_col].fillna("").astype(str).str.strip()
res_core = res_txt.str.replace(r'^[<>]\s*', '', regex=True).str.replace(",", "", regex=False)
res_num = pd.to_numeric(res_core, errors="coerce")

# derive detect (no explicit detect field)
nd_text = res_txt.str.contains(r'\bND\b|NON[- ]?DETECT|NOT DETECT|BDL', case=False, regex=True, na=False)
lt_flag = res_txt.str.startswith("<")
is_detect_y = (~nd_text) & (~lt_flag) & res_num.notna()    # numeric uncensored result
is_detect_n = nd_text | lt_flag                             # non-detect/censored

has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)

# ---- Table 1: ternary D/N/J/NF ----
state = np.where(
    is_detect_y & has_j, "J",
    np.where(is_detect_y & ~has_j, "D",
             np.where(is_detect_n, "N", "NF"))
)

tri = pd.DataFrame({"sample": sample, "analyte": analyte, "state": state})
tri.loc[(tri["sample"]=="") | (tri["analyte"]==""), "state"] = "NF"

priority = {"J":4, "D":3, "N":2, "NF":1}
tri["prio"] = tri["state"].map(priority).fillna(0).astype(int)
tri = tri.sort_values("prio", ascending=False).drop_duplicates(["sample","analyte"], keep="first")

ternary = tri.pivot(index="sample", columns="analyte", values="state").fillna("NF")
ternary = ternary.loc[ternary.index != "", ternary.columns != ""].sort_index().sort_index(axis=1)

# ---- Table 2: PQL bracket by year x analyte ----
year = pd.to_datetime(df[date_col], errors="coerce").dt.year
year = year.fillna(pd.to_numeric(df[date_col], errors="coerce")).astype("Int64")

pql_work = pd.DataFrame({
    "year": year,
    "analyte": analyte,
    "result_num": res_num,
    "is_j_detect": is_detect_y & has_j & res_num.notna(),
    "is_nq_detect": is_detect_y & (~has_j) & res_num.notna(),
})
pql_work = pql_work[(pql_work["year"].notna()) & (pql_work["analyte"]!="")]

agg = pql_work.groupby(["year","analyte"]).apply(
    lambda g: pd.Series({
        "highest_J": g.loc[g["is_j_detect"], "result_num"].max() if g["is_j_detect"].any() else np.nan,
        "lowest_NQ": g.loc[g["is_nq_detect"], "result_num"].min() if g["is_nq_detect"].any() else np.nan
    })
).reset_index()

pql_bracket = agg.pivot(index="year", columns="analyte", values=["highest_J","lowest_NQ"]).sort_index().sort_index(axis=1)

# save
stem = IN_PATH.stem.replace(" ","_")
tri_path = RUN_DIR / f"{stem}__ternary_DNJNF_matrix.csv"
pql_path = RUN_DIR / f"{stem}__PQL_bracket_highJ_lowNQ_by_year.csv"
ternary.to_csv(tri_path)
pql_bracket.to_csv(pql_path)

print("Saved:", tri_path)
print("Saved:", pql_path)
display(ternary.head(10))
display(pql_bracket.head(10))

Saved: C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\copy_of_R-80_zonal_data__ternary_DNJNF_matrix.csv
Saved: C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\copy_of_R-80_zonal_data__PQL_bracket_highJ_lowNQ_by_year.csv


C:\Users\kylian.robinson\AppData\Local\Temp\ipykernel_15468\3354975987.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)
C:\Users\kylian.robinson\AppData\Local\Temp\ipykernel_15468\3354975987.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  year = pd.to_datetime(df[date_col], errors="coerce").dt.year


analyte,Alkalinity,"Alkalinity, Total as CaCO3","Alkalinity, phenolphthalein",Aluminum,Antimony,Arsenic,Barium,Beryllium,Bicarbonate,Bicarbonate alkalinity (CaCO3),...,Silver,Sodium,Sulfate,Temperature,Thallium,Total Dissolved Solids,Total Organic Carbon,Uranium,Vanadium,Zinc
sample,,,,,,,,,,,,,,,,,,,,,
Zone 1,D,NF,NF,D,J,D,D,D,D,NF,...,D,D,D,D,D,D,D,D,D,D
Zone 2,D,NF,NF,D,J,D,D,D,D,NF,...,D,D,D,D,D,D,D,D,J,D
Zone 3,D,NF,NF,D,D,D,D,D,D,NF,...,D,D,D,D,D,D,D,D,D,D
Zone 4,NF,D,D,D,D,D,D,D,NF,D,...,D,D,D,D,D,D,D,D,D,D
Zone 5,NF,D,D,D,D,D,D,D,NF,D,...,D,D,D,D,D,D,D,D,D,D
Zone 6,NF,D,D,D,J,D,D,D,NF,D,...,D,D,D,D,D,D,D,D,J,D
Zone 7,NF,D,D,D,J,D,D,D,NF,D,...,D,D,D,D,D,D,D,D,J,J


highest_J                                                  \
analyte Alkalinity Bicarbonate Bromide Carbonate Chloride Fluoride   
year                                                                 
46178          NaN         NaN     NaN       NaN      NaN      NaN   
46180          NaN         NaN     NaN       NaN      NaN      NaN   
46181          NaN         NaN     NaN       NaN      NaN      NaN   
46182          NaN         NaN     NaN       NaN      NaN      NaN   
46184          NaN         NaN     NaN       NaN      NaN      NaN   
46185          NaN         NaN     NaN       NaN      NaN      NaN   
46186          NaN         NaN     NaN       NaN      NaN      NaN   

                                                                    \
analyte Nitrate-Nitrite as Nitrogen Sulfate Total Dissolved Solids   
year                                                                 
46178                          0.35     NaN                    NaN   
46180                          0.34     NaN                    NaN   
46181                          0.70     NaN                    NaN   
46182                           NaN     NaN                    NaN   
46184                           NaN     NaN                    NaN   
46185                          0.44     NaN                    NaN   
46186                          0.18     NaN                    NaN   

                              lowest_NQ                                \
analyte Total Organic Carbon Alkalinity Bicarbonate Bromide Carbonate   
year                                                                    
46178                    NaN       66.9        66.9    0.05       2.0   
46180                    NaN       70.0        70.0    0.05       2.0   
46181                    NaN       69.0        69.0    0.05       2.0   
46182                    NaN        NaN         NaN     NaN       NaN   
46184                    NaN        NaN         NaN     NaN       NaN   
46185                    NaN        NaN         NaN     NaN       NaN   
46186                    NaN        NaN         NaN     NaN       NaN   

                                                               \
analyte Chloride Fluoride Nitrate-Nitrite as Nitrogen Sulfate   
year                                                            
46178        2.4     0.27                         NaN     2.1   
46180        2.7     0.25                         NaN     2.3   
46181        2.9     0.27                         NaN     2.9   
46182        NaN      NaN                        0.61     NaN   
46184        NaN      NaN                        0.51     NaN   
46185        NaN      NaN                         NaN     NaN   
46186        NaN      NaN                         NaN     NaN   

                                                     
analyte Total Dissolved Solids Total Organic Carbon  
year                                                 
46178                    130.0                 0.46  
46180                    140.0                 0.46  
46181                    160.0                 0.46  
46182                      NaN                 0.46  
46184                      NaN                 0.46  
46185                      NaN                 0.46  
46186                      NaN                 3.30

In [8]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# ONE-CELL INTAKE CHECKPOINT
# outputs:
#   1) ternary matrix (D/N/J/NF) by sample x analyte
#   2) PQL bracket table by year x analyte with adjacent subcols:
#         analyte -> [highest_J, lowest_NQ]
# ============================================================

RUN_DIR = Path(r"C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun")
IN_PATH = RUN_DIR / "copy of R-80 zonal data.xlsx"   # change per block
SHEET_NAME = 0                                        # for xlsx

# ---------- load ----------
if IN_PATH.suffix.lower() in [".xlsx", ".xls"]:
    raw = pd.read_excel(IN_PATH, sheet_name=SHEET_NAME, dtype=str)
else:
    raw = pd.read_csv(IN_PATH, dtype=str, low_memory=False)

raw.columns = [str(c).strip() for c in raw.columns]

def norm(s):
    return re.sub(r'[^a-z0-9]+', '_', str(s).lower()).strip('_')

nmap = {c: norm(c) for c in raw.columns}

def find_one(patterns):
    rx = re.compile("|".join(patterns), re.I)
    for c in raw.columns:
        if rx.search(c) or rx.search(nmap[c]):
            return c
    return None

# ---------- schema mapping (handles both EIM + zonal-ish lab formats) ----------
sample_col   = find_one([r'^field_sample_id$', r'^sample(_id)?$', r'sample.*id', r'field.*id', r'location.*id', r'station.*id', r'^site$'])
analyte_col  = find_one([r'^parameter_name$', r'^analyte$', r'constituent', r'parameter', r'chemical'])
result_col   = find_one([r'^lab_result$', r'final_result', r'reported_result', r'(^|_)result($|_)', r'conc', r'value'])
valqual_col  = find_one([r'^lab_qualifier$', r'validation_qualifier', r'validation.*qual', r'validator_qual', r'val.*qual', r'qualifier', r'^qual$'])
detect_col   = find_one([r'^detect$', r'detected', r'detect_flag'])  # may be None (derive from result)
date_col     = find_one([r'^analysis_date$', r'^sample_date$', r'^year$', r'collection_date', r'collected', r'analysis.*date', r'date'])

print("Mapped columns:")
print(" sample  :", sample_col)
print(" analyte :", analyte_col)
print(" result  :", result_col)
print(" valqual :", valqual_col)
print(" detect  :", detect_col, "(optional)")
print(" date    :", date_col)

required = {"sample": sample_col, "analyte": analyte_col, "result": result_col, "valqual": valqual_col, "date": date_col}
missing = [k for k,v in required.items() if v is None]
if missing:
    raise ValueError(f"Missing required mapped columns: {missing}\nAvailable columns:\n{list(raw.columns)}")

df = raw.copy()

# ---------- common parse ----------
sample  = df[sample_col].fillna("").astype(str).str.strip()
analyte = df[analyte_col].fillna("").astype(str).str.strip()
vqual   = df[valqual_col].fillna("").astype(str).str.upper()

res_txt  = df[result_col].fillna("").astype(str).str.strip()
res_core = res_txt.str.replace(r'^[<>]\s*', '', regex=True).str.replace(",", "", regex=False)
res_num  = pd.to_numeric(res_core, errors="coerce")

# detect logic:
# - if detect column exists, use it
# - else derive:
#     N when ND/non-detect/BDL or '<'
#     Y when numeric uncensored
if detect_col is not None:
    d = df[detect_col].fillna("").astype(str).str.strip().str.upper()
    is_detect_y = d.isin(["Y","YES","TRUE","T","1"])
    is_detect_n = d.isin(["N","NO","FALSE","F","0"])
else:
    nd_text = res_txt.str.contains(r'\bND\b|NON[- ]?DETECT|NOT DETECT|BDL', case=False, regex=True, na=False)
    lt_flag = res_txt.str.startswith("<")
    is_detect_y = (~nd_text) & (~lt_flag) & res_num.notna()
    is_detect_n = nd_text | lt_flag

has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)

# ============================================================
# TABLE 1: TERNARY MATRIX (D/N/J/NF)
# ============================================================
state = np.where(
    is_detect_y & has_j, "J",
    np.where(
        is_detect_y & ~has_j, "D",
        np.where(is_detect_n, "N", "NF")
    )
)

tri = pd.DataFrame({"sample": sample, "analyte": analyte, "state": state})
tri.loc[(tri["sample"]=="") | (tri["analyte"]==""), "state"] = "NF"

# dedupe per sample+analyte: J > D > N > NF
priority = {"J":4, "D":3, "N":2, "NF":1}
tri["prio"] = tri["state"].map(priority).fillna(0).astype(int)
tri = tri.sort_values("prio", ascending=False).drop_duplicates(["sample","analyte"], keep="first")

ternary = tri.pivot(index="sample", columns="analyte", values="state").fillna("NF")
ternary = ternary.loc[ternary.index != "", ternary.columns != ""].sort_index(axis=0).sort_index(axis=1)

# ============================================================
# TABLE 2: PQL BRACKET OUTLOOK (year x analyte)
# columns are analyte-first with adjacent subcolumns:
#   analyte -> highest_J, lowest_NQ
# ============================================================

# parse year; prefers date parse, fallback to 4-digit extract
year = pd.to_datetime(df[date_col], errors="coerce").dt.year
year_fallback = pd.to_numeric(
    df[date_col].fillna("").astype(str).str.extract(r'((?:19|20)\d{2})', expand=False),
    errors="coerce"
)
year = year.fillna(year_fallback).astype("Int64")

# If all observed years are 2026, keep 2026 only (your zonal block case)
observed_years = sorted([int(y) for y in year.dropna().unique().tolist()])
if observed_years == [2026]:
    year_mask = (year == 2026)
else:
    year_mask = year.notna()   # keep all years for mixed blocks

pql_work = pd.DataFrame({
    "year": year,
    "analyte": analyte,
    "result_num": res_num,
    "is_j_detect": (is_detect_y & has_j & res_num.notna()),
    "is_nq_detect": (is_detect_y & (~has_j) & res_num.notna()),
})
pql_work = pql_work[year_mask & pql_work["analyte"].ne("")]

agg = (
    pql_work.groupby(["year","analyte"], dropna=False)
    .apply(lambda g: pd.Series({
        "highest_J": g.loc[g["is_j_detect"], "result_num"].max() if g["is_j_detect"].any() else np.nan,
        "lowest_NQ": g.loc[g["is_nq_detect"], "result_num"].min() if g["is_nq_detect"].any() else np.nan,
    }))
    .reset_index()
)

# ensure full analyte coverage from table 1 so no analyte silently disappears
all_analytes = pd.Index(ternary.columns, name="analyte")
all_years = sorted([int(y) for y in pql_work["year"].dropna().unique().tolist()])
if len(all_years) == 0:
    all_years = [2026]  # fallback display year if parse is empty
idx = pd.MultiIndex.from_product([all_years, all_analytes], names=["year","analyte"])

agg_full = (
    agg.set_index(["year","analyte"])
       .reindex(idx)
       .reset_index()
)

wide = agg_full.pivot(index="year", columns="analyte", values=["highest_J","lowest_NQ"])

# put analyte first, metrics second (adjacent pair per analyte)
wide = wide.swaplevel(0, 1, axis=1).sort_index(axis=1, level=[0,1])

# enforce metric order within each analyte
ordered_cols = []
for a in wide.columns.get_level_values(0).unique():
    for m in ["highest_J", "lowest_NQ"]:
        if (a, m) in wide.columns:
            ordered_cols.append((a, m))
wide = wide[ordered_cols]

# ============================================================
# SAVE
# ============================================================
stem = IN_PATH.stem.replace(" ", "_")
tri_path = RUN_DIR / f"{stem}__ternary_DNJNF_matrix.csv"
pql_path = RUN_DIR / f"{stem}__PQL_bracket_by_year_analyte_adjacent.csv"

ternary.to_csv(tri_path)
wide.to_csv(pql_path)

print("\nSaved:")
print(" ", tri_path)
print(" ", pql_path)

print("\nShapes:")
print(" table1 ternary:", ternary.shape)
print(" table2 pql    :", wide.shape)
print(" observed_years:", observed_years if len(observed_years) else "none parsed")

display(ternary.head(8))
display(wide.head(8))

Mapped columns:
 sample  : FIELD_SAMPLE_ID
 analyte : PARAMETER_CODE
 result  : RESULT_TYPE_CODE
 valqual : LAB_QUALIFIER
 detect  : None (optional)
 date    : ANALYSIS_DATE

Saved:
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\copy_of_R-80_zonal_data__ternary_DNJNF_matrix.csv
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\copy_of_R-80_zonal_data__PQL_bracket_by_year_analyte_adjacent.csv

Shapes:
 table1 ternary: (7, 43)
 table2 pql    : (1, 86)
 observed_years: [2026]


C:\Users\kylian.robinson\AppData\Local\Temp\ipykernel_15468\1092805537.py:85: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_j = vqual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)


analyte,AG,AL,ALK,ALK-CO3,ALK-CO3+HCO3,ALK-HCO3,ALK-OH,ALK-PHEN,AS,B,...,SB,SE,SO4(-2),TDS,TEMP,TL,TOC,U,V,ZN
sample,,,,,,,,,,,,,,,,,,,,,
Zone 1,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
Zone 2,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
Zone 3,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
Zone 4,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
Zone 5,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
Zone 6,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF
Zone 7,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF,...,NF,NF,NF,NF,NF,NF,NF,NF,NF,NF


analyte        AG                  AL                 ALK             ALK-CO3  \
        highest_J lowest_NQ highest_J lowest_NQ highest_J lowest_NQ highest_J   
year                                                                            
2026          NaN       NaN       NaN       NaN       NaN       NaN       NaN   

analyte           ALK-CO3+HCO3            ...        TL                 TOC  \
        lowest_NQ    highest_J lowest_NQ  ... highest_J lowest_NQ highest_J   
year                                      ...                                 
2026          NaN          NaN       NaN  ...       NaN       NaN       NaN   

analyte                   U                   V                  ZN            
        lowest_NQ highest_J lowest_NQ highest_J lowest_NQ highest_J lowest_NQ  
year                                                                           
2026          NaN       NaN       NaN       NaN       NaN       NaN       NaN  

[1 rows x 86 columns]

In [9]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

RUN_DIR = Path(r"C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun")
IN_PATH = RUN_DIR / "copy of R-80 zonal data.xlsx"
SHEET_NAME = 0

# ---------- Load ----------
if IN_PATH.suffix.lower() in [".xlsx", ".xls"]:
    raw = pd.read_excel(IN_PATH, sheet_name=SHEET_NAME, dtype=str)
else:
    raw = pd.read_csv(IN_PATH, dtype=str, low_memory=False)

raw.columns = [str(c).strip() for c in raw.columns]

# ---------- REQUIRED COLUMNS FOR THIS LAB FORMAT ----------
required_cols = [
    "FIELD_SAMPLE_ID", "PARAMETER_CODE", "PARAMETER_NAME",
    "LAB_RESULT", "LAB_QUALIFIER", "ANALYSIS_DATE"
]
missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw.copy()

# Canonical analyte key = PARAMETER_CODE (prevents name drift)
sample = df["FIELD_SAMPLE_ID"].fillna("").astype(str).str.strip()
pcode  = df["PARAMETER_CODE"].fillna("").astype(str).str.strip().str.upper()
pname  = df["PARAMETER_NAME"].fillna("").astype(str).str.strip()
analyte_key = pcode

# Optional label map (code -> most common name in this file)
label_map = (
    pd.DataFrame({"code": pcode, "name": pname})
    .query("code != ''")
    .groupby("code")["name"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
    .to_dict()
)

# ---------- Parse result / detect ----------
res_txt = df["LAB_RESULT"].fillna("").astype(str).str.strip()
res_core = res_txt.str.replace(r'^[<>]\s*', '', regex=True).str.replace(",", "", regex=False)
res_num = pd.to_numeric(res_core, errors="coerce")

qual = df["LAB_QUALIFIER"].fillna("").astype(str).str.upper()

# derive detect since no explicit detect column
nd_text = res_txt.str.contains(r'\bND\b|NON[- ]?DETECT|NOT DETECT|BDL', case=False, regex=True, na=False)
lt_flag = res_txt.str.startswith("<")
is_detect_y = (~nd_text) & (~lt_flag) & res_num.notna()
is_detect_n = nd_text | lt_flag
has_j = qual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)

# ---------- TABLE 1: Ternary D/N/J/NF (sample x PARAMETER_CODE) ----------
state = np.where(
    is_detect_y & has_j, "J",
    np.where(is_detect_y & ~has_j, "D",
             np.where(is_detect_n, "N", "NF"))
)

t1_long = pd.DataFrame({
    "sample": sample,
    "analyte": analyte_key,
    "state": state
})

# only analytes that actually exist in THIS file
t1_long = t1_long[(t1_long["sample"] != "") & (t1_long["analyte"] != "")]

# dedupe within sample+analyte using J > D > N > NF
priority = {"J":4, "D":3, "N":2, "NF":1}
t1_long["prio"] = t1_long["state"].map(priority).fillna(0).astype(int)
t1_long = (
    t1_long.sort_values("prio", ascending=False)
           .drop_duplicates(["sample", "analyte"], keep="first")
)

table1 = t1_long.pivot(index="sample", columns="analyte", values="state").fillna("NF")
table1 = table1.sort_index(axis=0).sort_index(axis=1)

# ---------- TABLE 2: 2026 PQL bracket outlook (by PARAMETER_CODE) ----------
# use ANALYSIS_DATE exactly as requested
year = pd.to_datetime(df["ANALYSIS_DATE"], errors="coerce").dt.year
year_txt = df["ANALYSIS_DATE"].fillna("").astype(str).str.extract(r'((?:19|20)\d{2})', expand=False)
year = year.fillna(pd.to_numeric(year_txt, errors="coerce")).astype("Int64")

# keep only 2026 rows for this block
m2026 = (year == 2026)

t2_long = pd.DataFrame({
    "year": year,
    "analyte": analyte_key,
    "result_num": res_num,
    "is_j_detect": is_detect_y & has_j & res_num.notna(),
    "is_nq_detect": is_detect_y & (~has_j) & res_num.notna(),
})
t2_long = t2_long[m2026 & (t2_long["analyte"] != "")]

agg = (
    t2_long.groupby(["year", "analyte"], dropna=False)
    .apply(lambda g: pd.Series({
        "highest_J": g.loc[g["is_j_detect"], "result_num"].max() if g["is_j_detect"].any() else np.nan,
        "lowest_NQ": g.loc[g["is_nq_detect"], "result_num"].min() if g["is_nq_detect"].any() else np.nan,
    }))
    .reset_index()
)

# Ensure analyte set exactly matches Table 1 analytes (this file only)
analytes_this_file = list(table1.columns)
if len(analytes_this_file) == 0:
    raise ValueError("No analytes found after parsing.")

idx = pd.MultiIndex.from_product([[2026], analytes_this_file], names=["year", "analyte"])
agg = agg.set_index(["year", "analyte"]).reindex(idx).reset_index()

# Build flat readable columns: AG_highest_J, AG_lowest_NQ, ...
wide = agg.pivot(index="year", columns="analyte", values=["highest_J", "lowest_NQ"])
wide = wide.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)

ordered_cols = []
for a in analytes_this_file:
    for m in ["highest_J", "lowest_NQ"]:
        if (a, m) in wide.columns:
            ordered_cols.append((a, m))
wide = wide.reindex(columns=ordered_cols)

# flatten columns for readability
wide.columns = [f"{a}_{m}" for a, m in wide.columns]
table2 = wide

# ---------- Save ----------
stem = IN_PATH.stem.replace(" ", "_")
t1_path = RUN_DIR / f"{stem}__table1_ternary_DNJNF_by_sample_parameter_code.csv"
t2_path = RUN_DIR / f"{stem}__table2_2026_PQL_bracket_AGstyle.csv"
map_path = RUN_DIR / f"{stem}__parameter_code_to_name_map.csv"

table1.to_csv(t1_path)
table2.to_csv(t2_path)
pd.DataFrame({"PARAMETER_CODE": list(label_map.keys()), "PARAMETER_NAME": list(label_map.values())}).to_csv(map_path, index=False)

print("Saved:")
print(" ", t1_path)
print(" ", t2_path)
print(" ", map_path)
print("\nShapes:")
print(" Table1:", table1.shape, "(sample x analyte)")
print(" Table2:", table2.shape, "(year x 2*analytes)")

display(table1.head(8))
display(table2.head(3))

Saved:
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\copy_of_R-80_zonal_data__table1_ternary_DNJNF_by_sample_parameter_code.csv
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\copy_of_R-80_zonal_data__table2_2026_PQL_bracket_AGstyle.csv
  C:\Users\kylian.robinson\yerPiCA\R-80_runs\MapRebuildItems\R80_ZonalRun\copy_of_R-80_zonal_data__parameter_code_to_name_map.csv

Shapes:
 Table1: (7, 41) (sample x analyte)
 Table2: (1, 82) (year x 2*analytes)


C:\Users\kylian.robinson\AppData\Local\Temp\ipykernel_15468\1960353462.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_j = qual.str.contains(r'(^|[^A-Z])J([^A-Z]|$)', regex=True, na=False)


analyte,AG,AL,ALK,ALK-CO3,ALK-CO3+HCO3,ALK-HCO3,ALK-OH,ALK-PHEN,AS,B,...,SB,SE,SO4(-2),TDS,TEMP,TL,TOC,U,V,ZN
sample,,,,,,,,,,,,,,,,,,,,,
Zone 1,D,D,D,NF,NF,NF,NF,NF,D,J,...,J,J,D,D,D,D,D,D,D,D
Zone 2,D,D,D,NF,NF,NF,NF,NF,D,J,...,J,J,D,D,D,D,D,D,J,D
Zone 3,D,D,D,NF,NF,NF,NF,NF,D,J,...,D,J,D,D,D,D,D,D,D,D
Zone 4,D,D,NF,D,D,D,D,D,D,J,...,D,J,D,D,D,D,D,D,D,D
Zone 5,D,D,NF,D,D,D,D,D,D,J,...,D,J,D,D,D,D,D,D,D,D
Zone 6,D,D,NF,D,D,D,D,D,D,J,...,J,D,D,D,D,D,D,D,J,D
Zone 7,D,D,NF,D,D,D,D,D,D,J,...,J,J,D,D,D,D,D,D,J,J


,AG_highest_J,AG_lowest_NQ,AL_highest_J,AL_lowest_NQ,ALK_highest_J,ALK_lowest_NQ,ALK-CO3_highest_J,ALK-CO3_lowest_NQ,ALK-CO3+HCO3_highest_J,ALK-CO3+HCO3_lowest_NQ,...,TL_highest_J,TL_lowest_NQ,TOC_highest_J,TOC_lowest_NQ,U_highest_J,U_lowest_NQ,V_highest_J,V_lowest_NQ,ZN_highest_J,ZN_lowest_NQ
year,,,,,,,,,,,,,,,,,,,,,
2026,NaN,1.0,NaN,19.3,NaN,66.9,NaN,0.725,NaN,64.1,...,NaN,5.0,NaN,0.46,NaN,0.503,4.85,5.16,4.35,3.3


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
base = Path(r"C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun")

R80_file = base / "Copy of R-80 zonal data.xlsx"
Intellus_file = base / "IntellusAo_R-80_Zonal.csv"
WOI_file = base / "WellsOfInterest.xls"

out_main = base / "ClosestToSynoptic_R80Zonal.csv"
out_summary = base / "ClosestToSynoptic_R80Zonal_EventSummary.csv"

# ------------------------------------------------------------------
# Read files
# ------------------------------------------------------------------
r80_raw = pd.read_excel(R80_file)
int_raw = pd.read_csv(Intellus_file, low_memory=False)
woi_raw = pd.read_excel(WOI_file)

# ------------------------------------------------------------------
# R80 file -> common schema (KEEP ALL ROWS, NO DROPS)
# ------------------------------------------------------------------
r80 = pd.DataFrame({
    "location": r80_raw["FIELD_SAMPLE_ID"].astype(str).str.strip(),
    "date": pd.to_datetime(r80_raw["ANALYSIS_DATE"], errors="coerce"),
    "analyte": r80_raw["PARAMETER_NAME"].astype(str).str.strip(),
    "value": pd.to_numeric(r80_raw["LAB_RESULT"], errors="coerce"),
    "unit": r80_raw["LAB_UNITS"].astype(str).str.strip(),
    "lab_qualifier": r80_raw["LAB_QUALIFIER"].astype(str).str.strip(),
    "mdl": pd.to_numeric(r80_raw["METHOD_DETECTION_LIMIT"], errors="coerce"),
    "source_file": "Copy of R-80 zonal data.xlsx"
})

# ND rule for R80 file
r80["detect"] = np.where(
    r80["value"].notna() & r80["mdl"].notna() & np.isclose(r80["value"], r80["mdl"], equal_nan=False),
    "N",
    "Y"
)

# ------------------------------------------------------------------
# Intellus file -> common schema
# ------------------------------------------------------------------
intellus = pd.DataFrame({
    "location": int_raw["Location"].astype(str).str.strip(),
    "date": pd.to_datetime(int_raw["Date Sampled"], errors="coerce"),
    "analyte": int_raw["Parameter"].astype(str).str.strip(),
    "value": pd.to_numeric(int_raw["Result"], errors="coerce"),
    "unit": int_raw["Units"].astype(str).str.strip(),
    "lab_qualifier": int_raw["Lab Qualifier"].astype(str).str.strip(),
    "detect": int_raw["Detect?"].astype(str).str.strip(),
    "source_file": "IntellusAo_R-80_Zonal.csv"
})

# Normalize detect flags in Intellus
intellus["detect"] = intellus["detect"].replace({
    "True": "Y", "False": "N",
    "TRUE": "Y", "FALSE": "N",
    "Yes": "Y", "No": "N",
    "YES": "Y", "NO": "N",
    "1": "Y", "0": "N"
})

# For Intellus only, keep rows with minimum keys needed for event selection
intellus = intellus.dropna(subset=["location", "date"])

# ------------------------------------------------------------------
# Filter Intellus ONLY by WellsOfInterest Location_ID substring (either direction)
# ------------------------------------------------------------------
patterns = (
    woi_raw["Location_ID"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)
patterns = [p for p in patterns if p]

def keep_by_substring_either_direction(loc):
    s = str(loc).strip().lower()
    return any((p in s) or (s in p) for p in patterns)

intellus_before = len(intellus)
intellus = intellus[intellus["location"].astype(str).apply(keep_by_substring_either_direction)].copy()

# ------------------------------------------------------------------
# Most recent Intellus sample event per location
# ------------------------------------------------------------------
event_summary = (
    intellus.groupby("location", as_index=False)["date"]
            .max()
            .rename(columns={"date": "effective_date"})
            .sort_values("location")
            .reset_index(drop=True)
)

intellus_latest = intellus.merge(
    event_summary,
    left_on=["location", "date"],
    right_on=["location", "effective_date"],
    how="inner"
).drop(columns=["effective_date"])

# ------------------------------------------------------------------
# Combine all R80 rows + latest Intellus rows
# ------------------------------------------------------------------
joined = pd.concat([r80, intellus_latest], ignore_index=True)

joined = joined[
    ["location", "date", "analyte", "value", "detect", "unit", "lab_qualifier", "source_file"]
].sort_values(["source_file", "location", "date", "analyte"]).reset_index(drop=True)

# ------------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------------
joined.to_csv(out_main, index=False)
event_summary.to_csv(out_summary, index=False)

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------
print(f"Saved synoptic file: {out_main}")
print(f"Saved event summary: {out_summary}")

print("\nReport:")
print(f"R80 rows read:                       {len(r80_raw):,}")
print(f"R80 rows kept (all):                 {len(r80):,}")
print(f"Intellus rows before WOI filter:     {intellus_before:,}")
print(f"Intellus rows after WOI filter:      {len(intellus):,}")
print(f"Intellus latest-event rows:          {len(intellus_latest):,}")
print(f"Combined output rows:                {len(joined):,}")

print("\nSynoptic preview:")
display(joined.head(20))

print("\nEvent summary preview:")
display(event_summary.head(20))

Saved synoptic file: C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\ClosestToSynoptic_R80Zonal.csv
Saved event summary: C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\ClosestToSynoptic_R80Zonal_EventSummary.csv

Report:
R80 rows read:                       297
R80 rows kept (all):                 297
Intellus rows before WOI filter:     175,382
Intellus rows after WOI filter:      133,064
Intellus latest-event rows:          25,144
Combined output rows:                25,441

Synoptic preview:


,location,date,analyte,value,detect,unit,lab_qualifier,source_file
0,Zone 1,2026-06-10,Alkalinity,67.00,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
1,Zone 1,2026-06-10,Alkalinity,66.90,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
2,Zone 1,2026-06-10,Bicarbonate,67.00,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
3,Zone 1,2026-06-10,Bicarbonate,66.90,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
4,Zone 1,2026-06-10,Carbonate,2.00,N,mg/L,U,Copy of R-80 zonal data.xlsx
5,Zone 1,2026-06-10,Carbonate,2.00,N,mg/L,U,Copy of R-80 zonal data.xlsx
6,Zone 1,2026-06-11,Total Dissolved Solids,130.00,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
7,Zone 1,2026-06-11,Total Organic Carbon,0.46,N,mg/L,U,Copy of R-80 zonal data.xlsx
8,Zone 1,2026-06-12,Bromide,0.05,N,mg/L,U,Copy of R-80 zonal data.xlsx
9,Zone 1,2026-06-12,Chloride,2.40,Y,mg/L,nan,Copy of R-80 zonal data.xlsx



Event summary preview:


,location,effective_date
0,CRPZ-1,2026-05-08
1,CRPZ-2A,2026-05-14
2,CRPZ-3,2026-05-08
3,CRPZ-4,2026-05-06
4,CRPZ-5,2026-05-14
5,CrEX-1,2026-05-27
6,CrEX-2,2026-05-27
7,CrEX-3,2026-05-28
8,CrEX-4,2026-05-27
9,CrIN-1,2025-04-24


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
base = Path(r"C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun")

R80_file = base / "Copy of R-80 zonal data.xlsx"
Intellus_file = base / "IntellusAo_R-80_Zonal.csv"
WOI_file = base / "WellsOfInterest.xls"

out_main = base / "ClosestToSynoptic_R80Zonal.csv"
out_summary = base / "ClosestToSynoptic_R80Zonal_EventSummary.csv"
out_mismatch = base / "ClosestToSynoptic_R80Zonal_LocationMismatchReport.csv"

# ------------------------------------------------------------------
# Read files
# ------------------------------------------------------------------
r80_raw = pd.read_excel(R80_file)
int_raw = pd.read_csv(Intellus_file, low_memory=False)
woi_raw = pd.read_excel(WOI_file)

# ------------------------------------------------------------------
# Build common schema for R80 file
# KEEP ALL ROWS from this file
# ------------------------------------------------------------------
r80 = pd.DataFrame({
    "location": r80_raw["FIELD_SAMPLE_ID"].astype(str).str.strip(),
    "date": pd.to_datetime(r80_raw["ANALYSIS_DATE"], errors="coerce"),
    "analyte": r80_raw["PARAMETER_NAME"].astype(str).str.strip(),
    "value": pd.to_numeric(r80_raw["LAB_RESULT"], errors="coerce"),
    "unit": r80_raw["LAB_UNITS"].astype(str).str.strip(),
    "lab_qualifier": r80_raw["LAB_QUALIFIER"].astype(str).str.strip(),
    "mdl": pd.to_numeric(r80_raw["METHOD_DETECTION_LIMIT"], errors="coerce"),
    "source_file": "Copy of R-80 zonal data.xlsx"
})

r80["detect"] = np.where(
    r80["value"].notna() & r80["mdl"].notna() & np.isclose(r80["value"], r80["mdl"], equal_nan=False),
    "N",
    "Y"
)

# ------------------------------------------------------------------
# Build common schema for Intellus file
# ------------------------------------------------------------------
intellus = pd.DataFrame({
    "location": int_raw["Location"].astype(str).str.strip(),
    "date": pd.to_datetime(int_raw["Date Sampled"], errors="coerce"),
    "analyte": int_raw["Parameter"].astype(str).str.strip(),
    "value": pd.to_numeric(int_raw["Result"], errors="coerce"),
    "unit": int_raw["Units"].astype(str).str.strip(),
    "lab_qualifier": int_raw["Lab Qualifier"].astype(str).str.strip(),
    "detect": int_raw["Detect?"].astype(str).str.strip(),
    "source_file": "IntellusAo_R-80_Zonal.csv"
})

intellus["detect"] = intellus["detect"].replace({
    "True": "Y", "False": "N",
    "TRUE": "Y", "FALSE": "N",
    "Yes": "Y", "No": "N",
    "YES": "Y", "NO": "N",
    "1": "Y", "0": "N"
})

# Only Intellus needs minimum keys for filtering/event selection
intellus = intellus.dropna(subset=["location", "date"])

# ------------------------------------------------------------------
# Wells of interest patterns
# ------------------------------------------------------------------
patterns = (
    woi_raw["Location_ID"]
    .dropna()
    .astype(str)
    .str.strip()
)
patterns = [p for p in patterns if p]

patterns_lower = [p.lower() for p in patterns]

def matching_patterns(loc):
    s = str(loc).strip().lower()
    hits = [p for p, p_lower in zip(patterns, patterns_lower) if (p_lower in s) or (s in p_lower)]
    return hits

# ------------------------------------------------------------------
# Mismatch / match report on Intellus locations
# ------------------------------------------------------------------
intellus_locations = sorted(intellus["location"].dropna().astype(str).str.strip().unique())

match_rows = []
for loc in intellus_locations:
    hits = matching_patterns(loc)
    match_rows.append({
        "intellus_location": loc,
        "matched": len(hits) > 0,
        "matched_location_id": " | ".join(hits) if hits else ""
    })

mismatch_report = pd.DataFrame(match_rows)
mismatch_report.to_csv(out_mismatch, index=False)

# ------------------------------------------------------------------
# Filter Intellus only to matched locations
# ------------------------------------------------------------------
matched_locations = set(
    mismatch_report.loc[mismatch_report["matched"], "intellus_location"]
)

intellus_before = len(intellus)
intellus = intellus[intellus["location"].isin(matched_locations)].copy()

# ------------------------------------------------------------------
# Select most recent Intellus sample event per location
# ------------------------------------------------------------------
event_summary = (
    intellus.groupby("location", as_index=False)["date"]
    .max()
    .rename(columns={"date": "effective_date"})
    .sort_values("location")
    .reset_index(drop=True)
)

intellus_latest = intellus.merge(
    event_summary,
    left_on=["location", "date"],
    right_on=["location", "effective_date"],
    how="inner"
).drop(columns=["effective_date"])

# ------------------------------------------------------------------
# Combine all R80 rows + latest Intellus rows
# ------------------------------------------------------------------
joined = pd.concat([r80, intellus_latest], ignore_index=True)

joined = joined[[
    "location", "date", "analyte", "value", "detect", "unit", "lab_qualifier", "source_file"
]].sort_values(["source_file", "location", "date", "analyte"]).reset_index(drop=True)

# ------------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------------
joined.to_csv(out_main, index=False)
event_summary.to_csv(out_summary, index=False)

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------
print(f"Saved synoptic file:      {out_main}")
print(f"Saved event summary:      {out_summary}")
print(f"Saved mismatch report:    {out_mismatch}")

print("\nCounts")
print(f"R80 rows kept (all):                      {len(r80):,}")
print(f"Intellus rows before location filter:     {intellus_before:,}")
print(f"Intellus rows after location filter:      {len(intellus):,}")
print(f"Intellus latest-event rows:               {len(intellus_latest):,}")
print(f"Combined output rows:                     {len(joined):,}")

print("\nLocation matching summary")
print(f"Unique Intellus locations checked:        {len(mismatch_report):,}")
print(f"Matched Intellus locations:               {mismatch_report['matched'].sum():,}")
print(f"Unmatched Intellus locations:             {(~mismatch_report['matched']).sum():,}")

print("\nUnmatched Intellus locations:")
display(mismatch_report.loc[~mismatch_report["matched"]].reset_index(drop=True))

print("\nMatched Intellus locations:")
display(mismatch_report.loc[mismatch_report["matched"]].reset_index(drop=True).head(30))

print("\nSynoptic preview:")
display(joined.head(20))

print("\nEvent summary preview:")
display(event_summary.head(20))

Saved synoptic file:      C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\ClosestToSynoptic_R80Zonal.csv
Saved event summary:      C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\ClosestToSynoptic_R80Zonal_EventSummary.csv
Saved mismatch report:    C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\ClosestToSynoptic_R80Zonal_LocationMismatchReport.csv

Counts
R80 rows kept (all):                      297
Intellus rows before location filter:     175,382
Intellus rows after location filter:      133,064
Intellus latest-event rows:               25,144
Combined output rows:                     25,441

Location matching summary
Unique Intellus locations checked:        305
Matched Intellus locations:               100
Unmatched Intellus locations:             205

Unmatched Intellus locations:


,intellus_location,matched,matched_location_id
0,18-MW-18,False,
1,50-24822 P142,False,
2,50-24822 P235,False,
3,50-24822 P25,False,
4,50-24822 P351,False,
...,...,...,...
200,PCI-2,False,
201,R-60,False,
202,R-80,False,
203,SCI-1,False,



Matched Intellus locations:


,intellus_location,matched,matched_location_id
0,CRPZ-1,True,CRPZ-1
1,CRPZ-2A,True,CRPZ-2A
2,CRPZ-3,True,CRPZ-3
3,CRPZ-4,True,CRPZ-4
4,CRPZ-5,True,CRPZ-5
5,CrEX-1,True,CrEX-1
6,CrEX-2,True,CrEX-2
7,CrEX-3,True,CrEX-3
8,CrEX-4,True,CrEX-4
9,CrIN-1,True,CrIN-1



Synoptic preview:


,location,date,analyte,value,detect,unit,lab_qualifier,source_file
0,Zone 1,2026-06-10,Alkalinity,67.00,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
1,Zone 1,2026-06-10,Alkalinity,66.90,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
2,Zone 1,2026-06-10,Bicarbonate,67.00,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
3,Zone 1,2026-06-10,Bicarbonate,66.90,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
4,Zone 1,2026-06-10,Carbonate,2.00,N,mg/L,U,Copy of R-80 zonal data.xlsx
5,Zone 1,2026-06-10,Carbonate,2.00,N,mg/L,U,Copy of R-80 zonal data.xlsx
6,Zone 1,2026-06-11,Total Dissolved Solids,130.00,Y,mg/L,nan,Copy of R-80 zonal data.xlsx
7,Zone 1,2026-06-11,Total Organic Carbon,0.46,N,mg/L,U,Copy of R-80 zonal data.xlsx
8,Zone 1,2026-06-12,Bromide,0.05,N,mg/L,U,Copy of R-80 zonal data.xlsx
9,Zone 1,2026-06-12,Chloride,2.40,Y,mg/L,nan,Copy of R-80 zonal data.xlsx



Event summary preview:


,location,effective_date
0,CRPZ-1,2026-05-08
1,CRPZ-2A,2026-05-14
2,CRPZ-3,2026-05-08
3,CRPZ-4,2026-05-06
4,CRPZ-5,2026-05-14
5,CrEX-1,2026-05-27
6,CrEX-2,2026-05-27
7,CrEX-3,2026-05-28
8,CrEX-4,2026-05-27
9,CrIN-1,2025-04-24


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
base = Path(r"C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun")
infile = base / "ClosestToSynoptic_R80Zonal.csv"
outfile = base / "R-80 data rotated.csv"

# ------------------------------------------------------------------
# Load synoptic long file
# Expected columns: location, date, analyte, value, detect, unit
# ------------------------------------------------------------------
df = pd.read_csv(infile)

# ------------------------------------------------------------------
# Clean basics
# ------------------------------------------------------------------
df["location"] = df["location"].astype(str).str.strip()
df["analyte"] = df["analyte"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df["unit"] = df["unit"].astype(str).str.strip()

# ------------------------------------------------------------------
# Normalize units text
# ------------------------------------------------------------------
def norm_unit(u):
    s = str(u).strip().lower()
    s = s.replace("µ", "u").replace("μ", "u")
    s = s.replace(" ", "")
    return s

df["_unit_norm"] = df["unit"].apply(norm_unit)

# ------------------------------------------------------------------
# Convert only selected analytes where units are dissimilar
# Ca, K, Mg, Na: ug/L -> mg/L ; mg/L unchanged
# ------------------------------------------------------------------
convert_analytes = {"Ca", "K", "Mg", "Na", "Calcium", "Potassium", "Magnesium", "Sodium"}

is_convert_analyte = df["analyte"].isin(convert_analytes)
is_ugL = df["_unit_norm"].isin({"ug/l", "ugl", "ug/l.", "µg/l", "μg/l"})
is_mgL = df["_unit_norm"].isin({"mg/l", "mgl", "mg/l."})

df.loc[is_convert_analyte & is_ugL, "value"] = df.loc[is_convert_analyte & is_ugL, "value"] / 1000.0
df.loc[is_convert_analyte & (is_ugL | is_mgL), "unit"] = "mg/L"

# ------------------------------------------------------------------
# Rotate long -> wide
# Use median if duplicate location/date/analyte rows exist
# ------------------------------------------------------------------
wide = (
    df.pivot_table(
        index=["location", "date"],
        columns="analyte",
        values="value",
        aggfunc="median"
    )
    .reset_index()
)

# flatten column index if needed
wide.columns.name = None

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
wide.to_csv(outfile, index=False)

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------
n_conv = int((is_convert_analyte & is_ugL).sum())

print(f"Loaded: {infile}")
print(f"Saved:  {outfile}")
print(f"Rows in long file: {len(df):,}")
print(f"Rows converted ug/L -> mg/L for Ca/K/Mg/Na: {n_conv:,}")
print(f"Rotated rows: {len(wide):,}")
print(f"Rotated columns: {len(wide.columns):,}")

print("\nPreview:")
display(wide.head(20))

Loaded: C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\ClosestToSynoptic_R80Zonal.csv
Saved:  C:\Users\kylian.robinson\MapRebuildItems\R80_ZonalRun\R-80 data rotated.csv
Rows in long file: 25,441
Rows converted ug/L -> mg/L for Ca/K/Mg/Na: 28
Rotated rows: 132
Rotated columns: 416

Preview:


,location,date,"1,3,5-Naphthalene trisulfonic acid","1,3,6-Naphthalene trisulfonic acid","1,5-Naphthalenedisulfonic acid","1,6-Naphthalene disulfonic acid",1-Naphthalene sulfonic acid,11-Chloroeicosafluoro-3-oxaundecane-1-sulfonic acid,"1H, 1H, 2H, 2H-Perfluorohexanesulfonic acid","1H, 1H, 2H, 2H-perfluorodecane sulfonic acid",...,Uranium-235/236,Uranium-238,Vanadium,Vinyl Chloride,Vinyl acetate,Xylene (Total),"Xylene[1,2-]","Xylene[1,3-]+Xylene[1,4-]",Zinc,n-Heptane
0,CRPZ-1,2026-05-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.950,NaN,NaN,NaN,NaN,NaN,3.300,NaN
1,CRPZ-2A,2026-05-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.280,NaN,NaN,NaN,NaN,NaN,6.270,NaN
2,CRPZ-3,2026-05-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,6.520,NaN,NaN,NaN,NaN,NaN,3.300,NaN
3,CRPZ-4,2026-05-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.120,NaN,NaN,NaN,NaN,NaN,3.300,NaN
4,CRPZ-5,2026-05-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.200,NaN,NaN,NaN,NaN,NaN,3.300,NaN
5,CrEX-1,2026-05-27,0.0020,0.00200,0.002,0.002,0.002,NaN,NaN,NaN,...,NaN,NaN,4.855,NaN,NaN,NaN,NaN,NaN,7.695,NaN
6,CrEX-2,2026-05-27,0.0020,0.00200,0.002,0.002,0.002,NaN,NaN,NaN,...,NaN,NaN,5.120,NaN,NaN,NaN,NaN,NaN,268.000,NaN
7,CrEX-3,2026-05-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CrEX-4,2026-05-27,0.0020,0.00200,0.002,0.002,0.002,NaN,NaN,NaN,...,NaN,NaN,6.040,NaN,NaN,NaN,NaN,NaN,8.250,NaN
9,CrIN-1,2025-04-24,0.0020,0.00200,0.002,0.002,0.002,NaN,NaN,NaN,...,NaN,NaN,4.315,NaN,NaN,NaN,NaN,NaN,24.600,NaN


In [8]:
##6/29 tweak version
#!/usr/bin/env python3
r"""
Pivot EDD long table to n×p matrix, then minimal housekeeping:
 - normalize concentration units in the LONG table using unit metadata,
 - ensure HCO3(-1) and CO3(-2) exist; compute Alkalinity-CO3+HCO3, then drop HCO3(-1) and CO3(-2),
 - rename columns using EDDtoIntellusNames.xlsx (EDD -> Intellus), ensuring NA -> Sodium,
 - drop ALK and TEMP columns from the pivot output,
 - drop NO3+NO2 and TOC if they have numeric values for only two samples.
"""

from pathlib import Path
import numpy as np
import pandas as pd

# --------- PATHS (edit these) ---------
ROOT_DIR = Path("C:/Users/kylian.robinson/MapRebuildItems")
INPUT_EDD_CSV = ROOT_DIR / "compile_odd_GC_forPCA.csv"
MAPPING_XLSX = ROOT_DIR / "EDDtoIntellusNames.xlsx"
OUTPUT_DIR = ROOT_DIR / "V 2.1 items"
OUTPUT_NAME = INPUT_EDD_CSV.stem + "_matrix_repV2.1.csv"
# --------------------------------------

def harmonic_mean(values: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    v = v[np.isfinite(v) & (v > 0)]
    if v.empty:
        return np.nan
    return float(len(v) / np.sum(1.0 / v))

def resolve_col(df: pd.DataFrame, candidates):
    low = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in low:
            return low[key]
    raise KeyError(f"Could not resolve any of {candidates} in columns: {list(df.columns)}")

def find_ci(cols, name):
    for c in cols:
        if c.strip().lower() == name.strip().lower():
            return c
    return None

def normalize_unit_text(x):
    if x is None:
        return ""
    s = str(x).strip().lower()
    s = s.replace("µ", "u").replace("μ", "u")
    s = s.replace(" ", "")
    return s

def convert_value_by_unit(value, unit_text):
    """
    Convert concentration to mg/L only when unit metadata shows a dissimilar unit.
    Leaves mg/L unchanged.
    """
    if pd.isna(value):
        return np.nan

    u = normalize_unit_text(unit_text)

    mgL_units = {"mg/l", "mgl", "mg/l."}
    ugL_units = {"ug/l", "ugl", "ug/l.", "µg/l", "μg/l"}

    if u in mgL_units:
        return value
    if u in ugL_units:
        return value / 1000.0

    # leave unchanged for unknown units; user can inspect later
    return value

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # 1) Load EDD (preserve literal strings like 'NA' in Parameter Code)
    df = pd.read_csv(INPUT_EDD_CSV, dtype=str, keep_default_na=False)

    field_col = resolve_col(df, ["field_sample_id", "sampleid", "sample_id"])
    param_col = resolve_col(df, ["parameter_code", "parameter", "parm_code"])
    result_col = resolve_col(df, ["lab_result", "result", "value"])
    unit_col = resolve_col(df, ["lab_units", "units", "unit", "result_unit"])

    # Clean parameter codes
    df[param_col] = df[param_col].astype(str).str.strip()
    blank_mask = (df[param_col] == "") | df[param_col].str.lower().isin({"nan", "none", "null"})
    if blank_mask.any():
        df = df.loc[~blank_mask].copy()

    # Coerce results
    df[result_col] = pd.to_numeric(df[result_col], errors="coerce")

    # 2) Normalize units in LONG table before aggregation/pivot
    df["value_mgL"] = [
        convert_value_by_unit(v, u)
        for v, u in zip(df[result_col], df[unit_col])
    ]

    # Optional reporting
    df["_unit_norm"] = df[unit_col].map(normalize_unit_text)
    print("Unit counts:")
    print(df["_unit_norm"].value_counts(dropna=False).sort_index())

    unknown_units = sorted(set(df["_unit_norm"]) - {"mg/l", "mgl", "mg/l.", "ug/l", "ugl", "ug/l.", ""})
    if unknown_units:
        print("\nUnits left unchanged because they were not recognized:")
        print(unknown_units)

    # 3) Aggregate duplicates per (sample, parameter) using harmonic mean
    agg = (
        df.groupby([field_col, param_col], dropna=False, sort=False)["value_mgL"]
          .apply(harmonic_mean)
          .reset_index()
    )

    # 4) Pivot to n×p matrix
    wide = (
        agg.pivot(index=field_col, columns=param_col, values="value_mgL")
           .sort_index(axis=0)
           .sort_index(axis=1)
    )

    # 5) Drop ALK and TEMP columns
    for drop_name in ["ALK", "TEMP"]:
        col = find_ci(wide.columns, drop_name)
        if col is not None:
            wide = wide.drop(columns=[col])

    # 6) Ensure HCO3(-1) and CO3(-2) exist, compute alkalinity surrogate
    if find_ci(wide.columns, "HCO3(-1)") is None:
        wide["HCO3(-1)"] = 0.0
    if find_ci(wide.columns, "CO3(-2)") is None:
        wide["CO3(-2)"] = 0.0

    hco3_col = find_ci(wide.columns, "HCO3(-1)")
    co3_col = find_ci(wide.columns, "CO3(-2)")

    wide[hco3_col] = pd.to_numeric(wide[hco3_col], errors="coerce").fillna(0.0)
    wide[co3_col] = pd.to_numeric(wide[co3_col], errors="coerce").fillna(0.0)
    wide["Alkalinity-CO3+HCO3"] = wide[hco3_col] + wide[co3_col]

    # 7) Drop bicarbonate/carbonate source columns
    wide = wide.drop(columns=[hco3_col, co3_col], errors="ignore")

    # 8) Rename columns using mapping file
    if MAPPING_XLSX.exists():
        map_df = pd.read_excel(MAPPING_XLSX, dtype=str)
        src = resolve_col(map_df, ["edd", "source", "from"])
        tgt = resolve_col(map_df, ["intellus", "intellus name", "name", "target", "to"])

        mapping = {}
        for _, row in map_df[[src, tgt]].dropna(subset=[src]).iterrows():
            k = str(row[src]).strip()
            v = str(row[tgt]).strip() if not pd.isna(row[tgt]) else k
            if k:
                mapping[k] = v or k

        if "NA" not in mapping:
            mapping["NA"] = "Sodium"

        wide = wide.rename(columns={c: mapping.get(c, c) for c in wide.columns})
        print("Applied EDD->Intellus renames (including NA->Sodium).")
    else:
        na_col = find_ci(wide.columns, "NA")
        if na_col is not None:
            wide = wide.rename(columns={na_col: "Sodium"})
        print(f"Mapping file not found: {MAPPING_XLSX}. Applied NA->Sodium fallback if needed.")

    # 9) Drop sparse NO3+NO2 and TOC
    for candidate in ["NO3+NO2", "TOC"]:
        col = find_ci(wide.columns, candidate)
        if col is not None:
            vals = pd.to_numeric(wide[col], errors="coerce")
            if int(vals.notna().sum()) == 2:
                wide = wide.drop(columns=[col])

    # 10) Save
    out_path = OUTPUT_DIR / OUTPUT_NAME
    wide.to_csv(out_path, na_rep="")
    print(f"Saved n×p matrix to: {out_path}")
    print(f"Rows (samples): {wide.shape[0]} | Columns (parameters): {wide.shape[1]}")

if __name__ == "__main__":
    main()

Unit counts:
_unit_norm
degc      9
mg/l     60
ug/l    268
Name: count, dtype: int64

Units left unchanged because they were not recognized:
['degc']
Applied EDD->Intellus renames (including NA->Sodium).
Saved n×p matrix to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\compile_odd_GC_forPCA_matrix_repV2.1.csv
Rows (samples): 7 | Columns (parameters): 34


In [7]:
#!/usr/bin/env python3
r"""
Pivot EDD long table to n×p matrix, then minimal housekeeping:
 - divide CA, K, MG, NA by 1000 (to mg/L),
 - ensure HCO3(-1) and CO3(-2) exist; compute Alkalinity-CO3+HCO3, then drop HCO3(-1) and CO3(-2),
 - rename columns using EDDtoIntellusNames.xlsx (EDD -> Intellus), ensuring NA -> Sodium,
 - drop ALK and TEMP columns from the pivot output,
 - drop NO3+NO2 and TOC if they have numeric values for only two samples.

Edit file paths here:
 - ROOT_DIR: base folder with inputs
 - INPUT_EDD_CSV: long EDD CSV (must have Field Sample ID, Parameter Code, Lab Result)
 - MAPPING_XLSX: Excel with two columns (EDD, Intellus)
 - OUTPUT_DIR: folder to write the n×p CSV into (V 2.1 items/)
"""

from pathlib import Path
import numpy as np
import pandas as pd

# --------- PATHS (edit these) ---------
ROOT_DIR = Path("C:/Users/kylian.robinson/MapRebuildItems")
INPUT_EDD_CSV = ROOT_DIR / "compile_odd_GC_forPCA.csv"          # long EDD-style table
MAPPING_XLSX = ROOT_DIR / "EDDtoIntellusNames.xlsx"              # two columns: EDD, Intellus
OUTPUT_DIR = ROOT_DIR / "V 2.1 items"                            # write n×p CSV here
OUTPUT_NAME = INPUT_EDD_CSV.stem + "_matrix_repV2.1.csv"                 # output file name
# --------------------------------------

# harmonic mean of positive finite values; NaN if none
def harmonic_mean(values: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    v = v[np.isfinite(v) & (v > 0)]
    if v.empty:
        return np.nan
    return float(len(v) / np.sum(1.0 / v))

def resolve_col(df: pd.DataFrame, candidates):
    low = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in low:
            return low[key]
    raise KeyError(f"Could not resolve any of {candidates} in columns: {list(df.columns)}")

def find_ci(cols, name):
    """Find actual column by case-insensitive exact match; return None if not found."""
    for c in cols:
        if c.strip().lower() == name.strip().lower():
            return c
    return None

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # 1) Load EDD (preserve literal strings like 'NA' in Parameter Code)
    df = pd.read_csv(INPUT_EDD_CSV, dtype=str, keep_default_na=False)
    field_col = resolve_col(df, ["field_sample_id", "sampleid", "sample_id"])
    param_col = resolve_col(df, ["parameter_code", "parameter", "parm_code"])
    result_col = resolve_col(df, ["lab_result", "result", "value"])

    # Clean parameter codes: strip; keep literal 'NA'; drop truly blank
    df[param_col] = df[param_col].astype(str).str.strip()
    blank_mask = (df[param_col] == "") | df[param_col].str.lower().isin({"nan", "none", "null"})
    if blank_mask.any():
        df = df.loc[~blank_mask].copy()

    # Coerce results to numeric for aggregation
    df[result_col] = pd.to_numeric(df[result_col], errors="coerce")

    # 2) Aggregate duplicates per (sample, parameter) using harmonic mean
    agg = (
        df.groupby([field_col, param_col], dropna=False, sort=False)[result_col]
          .apply(harmonic_mean)
          .reset_index()
    )

    # 3) Pivot to n×p matrix (rows=Field Sample ID, cols=Parameter Code)
    wide = (
        agg.pivot(index=field_col, columns=param_col, values=result_col)
           .sort_index(axis=0)
           .sort_index(axis=1)
    )

    # 4) Drop ALK and TEMP columns from pivot output (if present)
    for drop_name in ["ALK", "TEMP"]:
        col = find_ci(wide.columns, drop_name)
        if col is not None:
            wide = wide.drop(columns=[col])

    # 5) Unit conversion: divide CA, K, MG, NA by 1000 -> mg/L (if columns present)
    for ion in ["CA", "K", "MG", "NA"]:
        col = find_ci(wide.columns, ion)
        if col is not None:
            wide[col] = pd.to_numeric(wide[col], errors="coerce") / 1000.0

    # 6) Ensure HCO3(-1) and CO3(-2) exist (zeros if missing), then compute Alkalinity-CO3+HCO3
    if find_ci(wide.columns, "HCO3(-1)") is None:
        wide["HCO3(-1)"] = 0.0
    if find_ci(wide.columns, "CO3(-2)") is None:
        wide["CO3(-2)"] = 0.0
    # Use the exact actual names again (case-insensitive)
    hco3_col = find_ci(wide.columns, "HCO3(-1)")
    co3_col = find_ci(wide.columns, "CO3(-2)")
    wide[hco3_col] = pd.to_numeric(wide[hco3_col], errors="coerce").fillna(0.0)
    wide[co3_col]  = pd.to_numeric(wide[co3_col],  errors="coerce").fillna(0.0)
    wide["Alkalinity-CO3+HCO3"] = wide[hco3_col] + wide[co3_col]

    # 7) Drop HCO3(-1) and CO3(-2) after calculation
    wide = wide.drop(columns=[hco3_col, co3_col], errors="ignore")

    # 8) Rename columns using EDDtoIntellusNames.xlsx (EDD -> Intellus), ensuring NA -> Sodium
    if MAPPING_XLSX.exists():
        map_df = pd.read_excel(MAPPING_XLSX, dtype=str)
        src = resolve_col(map_df, ["edd", "source", "from"])
        tgt = resolve_col(map_df, ["intellus", "intellus name", "name", "target", "to"])
        mapping = {}
        for _, row in map_df[[src, tgt]].dropna(subset=[src]).iterrows():
            k = str(row[src]).strip()
            v = str(row[tgt]).strip() if not pd.isna(row[tgt]) else k
            if k:
                mapping[k] = v or k
        # Enforce NA -> Sodium even if mapping lacks it
        if "NA" not in mapping:
            mapping["NA"] = "Sodium"
        # Apply mapping to current columns (only exact matches)
        wide = wide.rename(columns={c: mapping.get(c, c) for c in wide.columns})
        print("Applied EDD->Intellus renames (including NA->Sodium).")
    else:
        # If no mapping, still enforce NA -> Sodium
        na_col = find_ci(wide.columns, "NA")
        if na_col is not None:
            wide = wide.rename(columns={na_col: "Sodium"})
        print(f"Mapping file not found: {MAPPING_XLSX}. Applied NA->Sodium fallback if needed.")

    # 9) Drop NO3+NO2 and TOC if they have numeric values for only two samples
    for candidate in ["NO3+NO2", "TOC"]:
        col = find_ci(wide.columns, candidate)
        if col is not None:
            vals = pd.to_numeric(wide[col], errors="coerce")
            non_null_count = int(vals.notna().sum())
            if non_null_count == 2:
                wide = wide.drop(columns=[col])

    # 10) Save n×p CSV (blank for missing)
    out_path = OUTPUT_DIR / OUTPUT_NAME
    wide.to_csv(out_path, na_rep="")
    print(f"Saved n×p matrix to: {out_path}")
    print(f"Rows (samples): {wide.shape[0]} | Columns (parameters): {wide.shape[1]}")

if __name__ == "__main__":
    main()

Applied EDD->Intellus renames (including NA->Sodium).
Saved n×p matrix to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\compile_odd_GC_forPCA_matrix_repV2.1.csv
Rows (samples): 7 | Columns (parameters): 34


In [12]:
#!/usr/bin/env python3
r"""
Build a treatment-plant parameter vector for location VS-CTUA-6-EFF using only
parameters that appear in the previously built matrix (compile_odd_GC_forPCA_matrix_repV2.1.csv).

Update:
 - The vector CSV now contains TWO rows:
     1) VS-CTUA-6-EFF | harmonic_mean | <parameters...>
     2) VS-CTUA-6-EFF | median        | <parameters...>
   where median is computed per-parameter over positive, finite values (consistent with the harmonic filter).
 - The first column is the Location ID ("VS-CTUA-6-EFF") and the second column is "Average Type".

Other outputs preserved:
 - watertreatment_VS-CTUA-6-EFF_vector_counts.csv (n positive obs per parameter for the harmonic mean)
 - watertreatment_VS-CTUA-6-EFF_records_by_date.csv (lossless pivot by sample date)
 - watertreatment_VS-CTUA-6-EFF_unmatched_parameters.csv (matrix params not observed in treatment EDD)
 - watertreatment_VS-CTUA-6-EFF_summary.json (small summary)

Paths (edit if needed):
  ROOT_DIR            = C:/Users/kylian.robinson/MapRebuildItems
  MATRIX_FILE         = compile_odd_GC_forPCA_matrix_repV2.1.csv  (under ROOT_DIR or V 2.1 items)
  TREATMENT_EDD_FILE  = Chemistry/watertreatment_9_1_24_ToDate.csv
  OUTPUT_DIR          = V 2.1 items

Harmonic mean: HM = n / Σ(1/x_i) over positive finite x_i; if none → NaN.
Median: per-parameter median over positive finite values; if none → NaN.
"""

from pathlib import Path
import pandas as pd
import numpy as np
import json

# -------------------- EDITABLE PATHS --------------------
ROOT_DIR = Path(r"C:/Users/kylian.robinson/MapRebuildItems")
MATRIX_FILE = ROOT_DIR / "compile_odd_GC_forPCA_matrix_repV2.1.csv"
ALT_MATRIX_FILE = ROOT_DIR / "V 2.1 items" / "compile_odd_GC_forPCA_matrix_repV2.1.csv"
TREATMENT_EDD_FILE = ROOT_DIR / "Chemistry" / "watertreatment_9_1_24_ToDate.csv"
OUTPUT_DIR = ROOT_DIR / "V 2.1 items"

OUTPUT_FILE_NAME = "watertreatment_VS-CTUA-6-EFF_vector.csv"
OUTPUT_COUNTS_NAME = "watertreatment_VS-CTUA-6-EFF_vector_counts.csv"
OUTPUT_RECORDS_BY_DATE_NAME = "watertreatment_VS-CTUA-6-EFF_records_by_date.csv"
OUTPUT_UNMATCHED_NAME = "watertreatment_VS-CTUA-6-EFF_unmatched_parameters.csv"
OUTPUT_SUMMARY_JSON = "watertreatment_VS-CTUA-6-EFF_summary.json"

LOCATION_TARGET = "VS-CTUA-6-EFF"
AVERAGE_TYPE_COL = "Average Type"
# --------------------------------------------------------


def harmonic_mean(values: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    v = v[np.isfinite(v) & (v > 0)]
    if v.empty:
        return np.nan
    return float(len(v) / np.sum(1.0 / v))


def median_positive(values: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    v = v[np.isfinite(v) & (v > 0)]
    if v.empty:
        return np.nan
    return float(np.median(v))


def count_positive(values: pd.Series) -> int:
    v = pd.to_numeric(values, errors="coerce")
    v = v[np.isfinite(v) & (v > 0)]
    return int(v.size)


def resolve_column(df: pd.DataFrame, candidates):
    low_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in low_map:
            return low_map[key]
    raise KeyError(f"Could not resolve any of {candidates} in columns: {list(df.columns)}")


def main():
    # --- Locate matrix file ---
    if not MATRIX_FILE.exists():
        if ALT_MATRIX_FILE.exists():
            matrix_path = ALT_MATRIX_FILE
        else:
            raise FileNotFoundError(
                f"Matrix file not found at {MATRIX_FILE} or {ALT_MATRIX_FILE}. "
                f"Build compile_odd_GC_forPCA_matrix_repV2.1.csv first."
            )
    else:
        matrix_path = MATRIX_FILE

    print(f"Using matrix file: {matrix_path}")
    matrix_df = pd.read_csv(matrix_path, dtype=str, keep_default_na=False)
    if matrix_df.shape[1] < 2:
        raise ValueError("Matrix file must have at least an ID column plus parameter columns.")
    # Parameter columns (exclude first assumed ID/sample column)
    matrix_parameters = set(matrix_df.columns[1:])
    print(f"Matrix parameter count: {len(matrix_parameters)}")

    # --- Load treatment plant EDD CSV ---
    if not TREATMENT_EDD_FILE.exists():
        raise FileNotFoundError(f"Treatment EDD file not found: {TREATMENT_EDD_FILE}")
    print(f"Loading treatment EDD: {TREATMENT_EDD_FILE}")
    edd_df = pd.read_csv(TREATMENT_EDD_FILE, dtype=str, keep_default_na=False)

    # Resolve needed columns
    loc_col = resolve_column(edd_df, ["location id", "location_id", "location", "loc id"])
    param_col = resolve_column(edd_df, ["parameter name", "parameter", "parameter_name"])
    result_col = resolve_column(edd_df, ["report result", "lab result", "result", "value"])
    date_col = resolve_column(edd_df, ["sample date", "date of sample", "collection date", "sampled date", "date"])

    # Clean
    edd_df[loc_col] = edd_df[loc_col].astype(str).str.strip()
    edd_df[param_col] = edd_df[param_col].astype(str).str.strip()
    edd_df[result_col] = pd.to_numeric(edd_df[result_col], errors="coerce")
    edd_df[date_col] = pd.to_datetime(edd_df[date_col], errors="coerce").dt.date

    # Filter for location and matrix parameters
    df_loc = edd_df[edd_df[loc_col].str.lower() == LOCATION_TARGET.lower()].copy()
    if df_loc.empty:
        raise ValueError(f"No rows for Location ID '{LOCATION_TARGET}'.")
    df_loc = df_loc[df_loc[param_col].isin(matrix_parameters)].copy()
    if df_loc.empty:
        raise ValueError("After filtering by matrix parameters, no data remain. Check naming alignment.")

    # Determine unmatched matrix parameters (matrix columns not present in this EDD selection)
    present_params = set(df_loc[param_col].unique())
    unmatched_params = sorted(matrix_parameters - present_params)
    print(f"Unmatched matrix parameters for {LOCATION_TARGET}: {len(unmatched_params)}")

    # Aggregate harmonic means, medians, and counts
    grouped = df_loc.groupby([loc_col, param_col], dropna=False, sort=False)[result_col]
    agg_h = grouped.apply(harmonic_mean).reset_index(name="harmonic_mean")
    agg_m = grouped.apply(median_positive).reset_index(name="median")
    agg_n = grouped.apply(count_positive).reset_index(name="n_obs")

    # Pivot to single-row (location as index), columns=parameters
    wide_h = agg_h.pivot(index=loc_col, columns=param_col, values="harmonic_mean").sort_index(axis=1)
    wide_m = agg_m.pivot(index=loc_col, columns=param_col, values="median").sort_index(axis=1)
    wide_n = agg_n.pivot(index=loc_col, columns=param_col, values="n_obs").sort_index(axis=1)

    # Ensure same columns/order
    all_params = sorted(set(wide_h.columns) | set(wide_m.columns))
    wide_h = wide_h.reindex(columns=all_params)
    wide_m = wide_m.reindex(columns=all_params)
    wide_n = wide_n.reindex(columns=all_params)

    # Build the two-row output with first column = Location ID, second = Average Type
    id_header = loc_col  # use the resolved location column name for clarity
    # Row for harmonic
    row_h = pd.DataFrame(
        [[LOCATION_TARGET, "harmonic_mean"] + [wide_h.loc[LOCATION_TARGET, c] for c in all_params]],
        columns=[id_header, AVERAGE_TYPE_COL] + all_params
    )
    # Row for median
    row_m = pd.DataFrame(
        [[LOCATION_TARGET, "median"] + [wide_m.loc[LOCATION_TARGET, c] for c in all_params]],
        columns=[id_header, AVERAGE_TYPE_COL] + all_params
    )
    vector_with_two_rows = pd.concat([row_h, row_m], ignore_index=True)

    # Full records by date (lossless): replicate columns if same date+parameter
    df_full = df_loc.dropna(subset=[date_col]).copy()
    df_full["_rep"] = df_full.groupby([date_col, param_col]).cumcount() + 1
    df_full["_colname"] = df_full.apply(
        lambda r: r[param_col] if r["_rep"] == 1 else f"{r[param_col]} (rep{r['_rep']})", axis=1
    )
    records_by_date = (
        df_full.pivot(index=date_col, columns="_colname", values=result_col)
               .sort_index(axis=0)
               .sort_index(axis=1)
    )
    records_by_date.index.name = "Sample Date"

    # Outputs
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_means = OUTPUT_DIR / OUTPUT_FILE_NAME
    out_counts = OUTPUT_DIR / OUTPUT_COUNTS_NAME
    out_records = OUTPUT_DIR / OUTPUT_RECORDS_BY_DATE_NAME
    out_unmatched = OUTPUT_DIR / OUTPUT_UNMATCHED_NAME
    out_summary = OUTPUT_DIR / OUTPUT_SUMMARY_JSON

    # vector with two rows (harmonic + median)
    vector_with_two_rows.to_csv(out_means, index=False, na_rep="")
    # counts remain single-row (n for the harmonic-mean filter set)
    wide_n.to_csv(out_counts, na_rep="")
    records_by_date.to_csv(out_records, na_rep="")
    pd.Series(unmatched_params, name="Unmatched_Parameters").to_csv(out_unmatched, index=False)

    summary = {
        "location": LOCATION_TARGET,
        "matrix_file": str(matrix_path),
        "treatment_file": str(TREATMENT_EDD_FILE),
        "vector_file": str(out_means),
        "counts_file": str(out_counts),
        "records_by_date_file": str(out_records),
        "unmatched_parameters_file": str(out_unmatched),
        "total_matrix_parameters": len(matrix_parameters),
        "retained_parameters": len(all_params),
        "unmatched_parameters_count": len(unmatched_params),
        "rows_in_vector_file": int(vector_with_two_rows.shape[0]),  # should be 2
        "average_types": vector_with_two_rows[AVERAGE_TYPE_COL].tolist(),
    }
    out_summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"\n✅ Vector (harmonic + median rows): {out_means}")
    print(f"✅ Counts (n): {out_counts}")
    print(f"✅ Records by date (lossless): {out_records}")
    print(f"✅ Unmatched parameter list: {out_unmatched}")
    print(f"✅ Summary JSON: {out_summary}")
    print(f"Vector rows: {vector_with_two_rows.shape[0]} (expected 2) | Parameters: {len(all_params)}")

if __name__ == "__main__":
    main()

Using matrix file: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\compile_odd_GC_forPCA_matrix_repV2.1.csv
Matrix parameter count: 33
Loading treatment EDD: C:\Users\kylian.robinson\MapRebuildItems\Chemistry\watertreatment_9_1_24_ToDate.csv
Unmatched matrix parameters for VS-CTUA-6-EFF: 1

✅ Vector (harmonic + median rows): C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\watertreatment_VS-CTUA-6-EFF_vector.csv
✅ Counts (n): C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\watertreatment_VS-CTUA-6-EFF_vector_counts.csv
✅ Records by date (lossless): C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\watertreatment_VS-CTUA-6-EFF_records_by_date.csv
✅ Unmatched parameter list: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\watertreatment_VS-CTUA-6-EFF_unmatched_parameters.csv
✅ Summary JSON: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\watertreatment_VS-CTUA-6-EFF_summary.json
Vector rows: 2 (expected 2) | Parameters: 32


In [13]:
#!/usr/bin/env python3
"""
Combine Drillsite_SelectedParams with the treatment plant vector (two-row: harmonic_mean + median)
and preserve a two-column identifier format:
  - Column 1: ID (same as the first column in the drill site file)
  - Column 2: Average Type
      * "Drillsite" for all drill rows
      * "harmonic_mean" and "median" for the two treatment rows

Behavior:
 1) Load (both in V 2.1 items):
      - Drillsite_SelectedParams.csv
      - watertreatment_VS-CTUA-6-EFF_vector.csv  (two rows: harmonic_mean + median)
 2) Sort drill parameter columns alphabetically (excluding ID and Average Type).
 3) Ensure treatment vector has first two columns: ID and 'Average Type' (rename if necessary).
 4) Force treatment ID label to 'VS-CTUA-6-EFF' (so it’s consistent).
 5) Align treatment columns to drill parameter set:
      - Add missing drill parameters to treatment rows as blank ("")
      - Drop treatment-only parameters not present in the drill set
 6) Insert 'Average Type' = 'Drillsite' for drill rows and place it as the second column.
 7) Append the two treatment rows to the drill rows.
 8) Save anthropogenic_data.csv in the same folder.

Run in Jupyter:
    %run build_anthropogenic_data.py

Editables:
 - BASE_DIR, filenames
 - ID_COL (force ID column name, or None to auto-use the first drill column)
 - AVERAGE_TYPE_COL name (defaults to 'Average Type')
 - TREATMENT_ID label ('VS-CTUA-6-EFF' by default)
"""

from pathlib import Path
import pandas as pd

# -------- Configuration --------
BASE_DIR = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items")
DRILL_FILE = BASE_DIR / "Drillsite_SelectedParams.csv"
TREATMENT_FILE = BASE_DIR / "watertreatment_VS-CTUA-6-EFF_vector.csv"
OUT_FILE = BASE_DIR / "anthropogenic_data.csv"

ID_COL = None                 # e.g., "Location ID"; None -> use first column of drill file
AVERAGE_TYPE_COL = "Average Type"
TREATMENT_ID = "VS-CTUA-6-EFF"  # enforced label for both treatment rows

def main():
    if not DRILL_FILE.exists():
        raise FileNotFoundError(f"Drill site file not found: {DRILL_FILE}")
    if not TREATMENT_FILE.exists():
        raise FileNotFoundError(f"Treatment vector file not found: {TREATMENT_FILE}")

    drill_df = pd.read_csv(DRILL_FILE, dtype=str, keep_default_na=False)
    treat_df = pd.read_csv(TREATMENT_FILE, dtype=str, keep_default_na=False)

    # Determine ID column from drill file
    id_col = ID_COL or drill_df.columns[0]
    if id_col not in drill_df.columns:
        raise ValueError(f"ID column '{id_col}' not found in drill file.")

    # Prepare drill data: ensure Average Type exists and is 2nd column; sort parameter columns
    drill = drill_df.copy()
    # If drill file already includes Average Type, keep it; otherwise add it
    if AVERAGE_TYPE_COL not in drill.columns:
        drill.insert(1, AVERAGE_TYPE_COL, "Drillsite")
    else:
        # Ensure all drill rows labeled consistently
        drill[AVERAGE_TYPE_COL] = "Drillsite"

    # Parameter columns for drill = all except ID and Average Type
    drill_param_cols = [c for c in drill.columns if c not in (id_col, AVERAGE_TYPE_COL)]
    drill_param_cols_sorted = sorted(drill_param_cols, key=lambda x: x.lower())

    # Reorder drill columns: [ID, Average Type, sorted params...]
    drill = drill[[id_col, AVERAGE_TYPE_COL] + drill_param_cols_sorted]

    # Prepare treatment vector: must have first two columns = ID + Average Type
    # Rename first two columns to match our schema if needed
    if treat_df.shape[1] < 2:
        raise ValueError("Treatment vector must have at least two columns (ID and Average Type) plus parameters.")

    treat_cols = list(treat_df.columns)
    # Rename first column to id_col if different
    if treat_cols[0] != id_col:
        treat_df = treat_df.rename(columns={treat_cols[0]: id_col})
    # Rename second column to AVERAGE_TYPE_COL if different
    if treat_df.columns[1] != AVERAGE_TYPE_COL:
        treat_df = treat_df.rename(columns={treat_df.columns[1]: AVERAGE_TYPE_COL})

    # Enforce treatment ID label
    treat_df[id_col] = TREATMENT_ID

    # Align treatment parameter columns to the drill parameter set:
    # - add missing columns as ""
    # - drop extra columns (not in drill set)
    treat_param_cols_current = [c for c in treat_df.columns if c not in (id_col, AVERAGE_TYPE_COL)]
    missing_for_treat = [c for c in drill_param_cols_sorted if c not in treat_param_cols_current]
    for col in missing_for_treat:
        treat_df[col] = ""
    # Keep only ID, Average Type, and drill params
    treat_df = treat_df[[id_col, AVERAGE_TYPE_COL] + drill_param_cols_sorted]

    # Combine: drill rows first, then both treatment rows
    combined = pd.concat([drill, treat_df], ignore_index=True)

    # Save combined
    combined.to_csv(OUT_FILE, index=False, na_rep="")

    # Report
    print(f"Saved combined file to: {OUT_FILE}")
    print(f"- Drill rows: {len(drill_df)}")
    print(f"- Treatment rows appended: {len(treat_df)} (expected 2: harmonic_mean, median)")
    print(f"- Identifier columns: [{id_col!r}, {AVERAGE_TYPE_COL!r}]")
    print(f"- Parameter columns (sorted): {len(drill_param_cols_sorted)}")
    if missing_for_treat:
        print(f"- Params missing in treatment (filled blank): {len(missing_for_treat)} "
              f"(first 15: {missing_for_treat[:15]})")

if __name__ == "__main__":
    main()

Saved combined file to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\anthropogenic_data.csv
- Drill rows: 7
- Treatment rows appended: 2 (expected 2: harmonic_mean, median)
- Identifier columns: ['FIELD_SAMPLE_ID', 'Average Type']
- Parameter columns (sorted): 32


In [20]:
#!/usr/bin/env python3
r"""
Intellus_matrix_builder
Build a location×date parameter matrix aligned to a predefined parameter set, taking the HIGHEST
(measured) value when there are multiple samples for the same (Location, Date, Parameter).

Update:
 - Removed the argument skipna=True from GroupBy.max(), which caused a TypeError in your pandas
   version (pandas GroupBy.max does not accept a skipna keyword there). The default behavior
   already ignores NaNs for numeric dtypes.

Process:
 1. Load manual parameter matrix (anthropogenic_data.csv) to determine final parameter columns.
 2. Load long-form chemistry file (CrWedge_Analytes_fr9_1_24_ao_10_21_25.csv).
 3. Resolve column names case-insensitively.
 4. Strip strings; keep literal tokens like 'NA' in parameter names.
 5. Drop truly blank parameter names (keep literal 'NA').
 6. Coerce result column to numeric (non-numeric -> NaN).
 7. Aggregate duplicate (Location, Date, Parameter) rows by maximum numeric value.
 8. Pivot to wide matrix (index = Location, Date).
 9. Reindex columns to manual parameter ordering (adding missing empty columns, dropping extras).
10. Write CSV with blanks for NaN.

Edit paths below if needed.
"""

from pathlib import Path
import pandas as pd
import numpy as np

# -------------------- CONFIGURABLE PATHS --------------------
MANUAL_PARAMS_PATH = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items/anthropogenic_data.csv")
CHEM_INPUT_PATH    = Path(r"C:/Users/kylian.robinson/MapRebuildItems/Chemistry/CrWedge_Analytes_fr9_1_24_ao_10_21_25.csv")
CHEM_OUTPUT_PATH   = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items/CrWedge_matrix_location_date.csv")
# ------------------------------------------------------------

def main():
    # 1. Manual parameter columns (order to enforce)
    manual_df = pd.read_csv(MANUAL_PARAMS_PATH, index_col=0, keep_default_na=False, dtype=str)
    manual_params = list(manual_df.columns)
    if not manual_params:
        raise ValueError(f"No parameter columns found in manual parameter matrix: {MANUAL_PARAMS_PATH}")

    # 2. Chemistry long-form
    chem_df = pd.read_csv(CHEM_INPUT_PATH, keep_default_na=False)

    # 3. Resolve columns
    lower_map = {c.lower().strip(): c for c in chem_df.columns}

    def resolve_col(possible_names, what):
        for name in possible_names:
            key = name.lower().strip()
            if key in lower_map:
                return lower_map[key]
        raise ValueError(
            f"Could not find required column for {what}. "
            f"Tried: {possible_names}. "
            f"Available: {list(chem_df.columns)}"
        )

    location_col = resolve_col(
        ["location", "location id", "location_id", "locationid", "site id", "site"],
        "Location"
    )
    date_col = resolve_col(
        ["date sampled", "sample date", "collection date", "sampled date", "date"],
        "Date Sampled"
    )
    param_col = resolve_col(
        ["parameter_code", "parameter name", "parameter", "analyte", "analyte name",
         "constituent", "analyte_name"],
        "Parameter"
    )
    result_col = resolve_col(
        ["lab_result", "report result", "result", "value", "result value",
         "reported value", "parameter result", "final result"],
        "Numeric Result"
    )

    # 4. Clean identifiers
    chem_df[location_col] = chem_df[location_col].astype(str).str.strip()
    chem_df[date_col]     = chem_df[date_col].astype(str).str.strip()
    chem_df[param_col]    = chem_df[param_col].astype(str).str.strip()

    # 5. Drop blank parameter names (keep literal 'NA')
    blank_param_mask = (chem_df[param_col] == "") | (chem_df[param_col].str.lower().isin({"", "nan", "none"}))
    dropped_blank = int(blank_param_mask.sum())
    if dropped_blank:
        chem_df = chem_df.loc[~blank_param_mask].copy()

    # 6. Coerce numeric results
    chem_df[result_col] = pd.to_numeric(chem_df[result_col], errors="coerce")

    # 7. Aggregate duplicates: maximum value per (Location, Date, Parameter)
    # Removed skipna=True because GroupBy.max() does not accept it in your pandas version.
    agg_df = (
        chem_df.groupby([location_col, date_col, param_col], dropna=False, sort=False)[result_col]
               .max()
               .reset_index()
    )

    # 8. Pivot
    wide = (
        agg_df.pivot(index=[location_col, date_col], columns=param_col, values=result_col)
              .sort_index(axis=0)
              .sort_index(axis=1)
    )

    # 9. Align to manual parameter set
    wide_aligned = wide.reindex(columns=manual_params)

    # 10. Output
    wide_out = wide_aligned.reset_index()
    wide_out = wide_out[[location_col, date_col] + manual_params]

    CHEM_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    wide_out.to_csv(CHEM_OUTPUT_PATH, index=False, na_rep="")

    extra_cols = [c for c in wide.columns if c not in manual_params]
    missing_cols = [c for c in manual_params if c not in wide.columns]

    print(f"✅ Input chemistry long-form: {CHEM_INPUT_PATH}")
    print(f"✅ Manual parameter matrix:  {MANUAL_PARAMS_PATH}")
    print(f"✅ Wrote aligned MAX matrix: {CHEM_OUTPUT_PATH}")
    print(f"   Rows (Location, Date): {wide_out.shape[0]} | Parameters (manual set): {len(manual_params)}")
    if dropped_blank:
        print(f"ℹ️ Dropped {dropped_blank} rows with blank parameter names (kept literal 'NA').")
    if extra_cols:
        print(f"ℹ️ Dropped {len(extra_cols)} parameters not in manual set (first 10): "
              f"{extra_cols[:10]}{'...' if len(extra_cols) > 10 else ''}")
    if missing_cols:
        print(f"ℹ️ Added {len(missing_cols)} empty columns from manual set absent in chemistry (first 10): "
              f"{missing_cols[:10]}{'...' if len(missing_cols) > 10 else ''}")

if __name__ == "__main__":
    main()

✅ Input chemistry long-form: C:\Users\kylian.robinson\MapRebuildItems\Chemistry\CrWedge_Analytes_fr9_1_24_ao_10_21_25.csv
✅ Manual parameter matrix:  C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\anthropogenic_data.csv
✅ Wrote aligned MAX matrix: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\CrWedge_matrix_location_date.csv
   Rows (Location, Date): 1026 | Parameters (manual set): 33
ℹ️ Dropped 391 parameters not in manual set (first 10): ['1,3,5-Naphthalene trisulfonic acid', '1,3,6-Naphthalene trisulfonic acid', '1,5-Naphthalenedisulfonic acid', '1,6-Naphthalene disulfonic acid', '1-Naphthalene sulfonic acid', '11-Chloroeicosafluoro-3-oxaundecane-1-sulfonic acid', '1H, 1H, 2H, 2H-Perfluorododecanesulphonic acid', '1H, 1H, 2H, 2H-Perfluorohexanesulfonic acid', '1H, 1H, 2H, 2H-perfluorodecane sulfonic acid', '1H, 1H, 2H, 2H-perfluorooctane sulfonic acid']...
ℹ️ Added 1 empty columns from manual set absent in chemistry (first 10): ['Type']


In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

# Default input: the aligned matrix with Location and Date Sampled columns
# If you saved with the earlier script, this should exist:
default_input = Path(r"C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\CrWedge_matrix_location_date.csv")

# If the exact filename changed, we’ll fall back to any *_matrix_manualcols_location_date.csv in the Chemistry folder
if not default_input.exists():
    chem_dir = default_input.parent
    candidates = sorted(chem_dir.glob("*_matrix_manualcols_location_date.csv"))
    if not candidates:
        raise FileNotFoundError(f"Could not find an input matrix like *_matrix_manualcols_location_date.csv in {chem_dir}")
    input_path = candidates[0]
else:
    input_path = default_input

# Output path
output_path = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items/CrWedge_AnalytesTruncated_MR.csv")

# Read the saved matrix; blanks will come back as NaN by default
df = pd.read_csv(input_path, keep_default_na=True)

# Resolve id columns (case-insensitive)
lower_map = {c.lower().strip(): c for c in df.columns}
def resolve_col(possible, what):
    for n in possible:
        if n in lower_map:
            return lower_map[n]
    raise ValueError(f"Missing required column for {what}. Looked for: {possible}. Found: {list(df.columns)}")

location_col = resolve_col(["location", "location id", "location_id", "site", "site id"], "Location")
date_col     = resolve_col(["date sampled", "sample date", "collection date", "date"], "Date Sampled")

# Parse dates; drop rows where date cannot be parsed
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=[date_col])

# Parameter columns are everything except the id columns
param_cols = [c for c in df.columns if c not in {location_col, date_col}]

# Coerce parameter columns to numeric to standardize completeness checks
for c in param_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Keep only rows with a complete parameter set (no NaNs across all parameter columns)
complete_mask = df[param_cols].notna().all(axis=1)
df_complete = df.loc[complete_mask].copy()

# For each location, select the most recent row (max date)
if df_complete.empty:
    raise ValueError("No rows have a complete parameter set; nothing to output.")

idx_latest = df_complete.groupby(location_col, as_index=False)[date_col].idxmax()[date_col]
latest_rows = df_complete.loc[idx_latest].copy()

# Sort by Location for readability and keep column order as in the input
latest_rows = latest_rows.sort_values(by=[location_col])

# Save to CSV with blanks for any (should be none) missing values
output_path.parent.mkdir(parents=True, exist_ok=True)
latest_rows.to_csv(output_path, index=False, na_rep="")

# Report summary
total_locs = df[location_col].nunique(dropna=True)
covered_locs = latest_rows[location_col].nunique(dropna=True)
print(f"✅ Input: {input_path}")
print(f"✅ Wrote most-recent complete rows to: {output_path}")
print(f"   Locations covered: {covered_locs} / {total_locs}")

✅ Input: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\CrWedge_matrix_location_date.csv
✅ Wrote most-recent complete rows to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\CrWedge_AnalytesTruncated_MR.csv
   Locations covered: 87 / 282


In [24]:
#!/usr/bin/env python3
r"""
Purpose: assign well-based categories (Cat) on verified date-specific data, and overwrite the Date
column in-place with Cat for downstream plotting.

What this does:
 1) Load the analyte matrix (expects Location and Date columns present).
 2) Load WellsOfInterest.xls and build category mappings using:
      - Exact Location_ID -> Cat
      - Exact Well_ID -> Cat
      - (BaseWell, screen rank) -> Cat via DepthEnd or explicit S# suffix
      - Singleton base-well fallback -> Cat
 3) Resolve Cat for each matrix row based on its Location.
 4) Overwrite the Date column (same position) with Cat.
 5) Any row that still lacks a Cat is assigned "Intermediate" (dataset-specific rule).
 6) Save an unmatched report listing rows that had no Cat BEFORE the "Intermediate" fill.

Notes:
 - Column order is preserved except that the Date column is replaced by a new column named "Cat"
   at the exact same position as Date.
"""

import re
from pathlib import Path
import pandas as pd
import numpy as np

# Paths
base_dir = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items/")
input_matrix = base_dir / "CrWedge_AnalytesTruncated_MR.csv"
wells_path = base_dir / "WellsOfInterest.xls"
output_matrix = base_dir / "CrWedge_AnalytesTruncated_MR_withCat.csv"
unmatched_out = base_dir / "CrWedge_Analytes_UnmatchedToCat.csv"

# Load truncated matrix (expects Location and Date columns present)
df = pd.read_csv(input_matrix, keep_default_na=True)

# Resolve key columns in matrix (case-insensitive, strip whitespace)
lower_map = {c.lower().strip(): c for c in df.columns}
def resolve_col(possible, what):
    for n in possible:
        if n in lower_map:
            return lower_map[n]
    raise ValueError(f"Missing required column for {what}. Looked for: {possible}. Found: {list(df.columns)}")

location_col = resolve_col(["location", "location id", "location_id", "site", "site id"], "Location")
date_col     = resolve_col(["date sampled", "sample date", "collection date", "date"], "Date Sampled")

# Require Aluminum to be present (kept from base script; not used for placement anymore)
al_col = None
for c in df.columns:
    if c.strip().lower() == "aluminum":
        al_col = c
        break
if al_col is None:
    raise ValueError("Expected an 'Aluminum' column in the matrix, but it was not found.")

# Load WellsOfInterest.xls with deterministic ID priority (Location_ID > Well_ID > Site_ID)
xls = pd.ExcelFile(wells_path)

def canon(s: str) -> str:
    s = str(s).lower()
    return "".join(ch for ch in s if ch.isalnum())

id_priority = [
    "location_id", "location id", "location",
    "well_id", "well id", "well",
    "site_id", "site id", "site"
]
cat_keys = ["cat"]
depth_candidates = [
    "perforation_zone_end_depth", "perforation zone end depth",
    "perforation_end_depth", "perforation end depth",
    "perforation_end", "perforation end",
    "screen_end_depth", "screen end depth",
    "screen_end", "screen end",
    "perforation_bottom", "perforation bottom",
    "perf_end_depth", "perf end depth",
]

wells_frames = []
for sheet in xls.sheet_names:
    w_raw = pd.read_excel(xls, sheet_name=sheet)
    cm = {canon(c): c for c in w_raw.columns}

    # Cat
    cat_col_src = None
    for k in cat_keys:
        if canon(k) in cm:
            cat_col_src = cm[canon(k)]
            break
    if not cat_col_src:
        continue

    # IDs (deterministic priority)
    loc_id_src = None; well_id_src = None; site_id_src = None
    for key in id_priority:
        ck = canon(key)
        if ck in cm:
            if "location" in key and loc_id_src is None:
                loc_id_src = cm[ck]
            elif "well" in key and well_id_src is None:
                well_id_src = cm[ck]
            elif "site" in key and site_id_src is None:
                site_id_src = cm[ck]

    if not any([loc_id_src, well_id_src, site_id_src]):
        continue

    # Depth (optional)
    depth_col_src = None
    for k in depth_candidates:
        ck = canon(k)
        if ck in cm:
            depth_col_src = cm[ck]
            break

    keep_cols = [c for c in [loc_id_src, well_id_src, site_id_src, cat_col_src, depth_col_src] if c]
    sub = w_raw[keep_cols].copy()

    rename_map = {}
    if loc_id_src:  rename_map[loc_id_src] = "Location_ID"
    if well_id_src: rename_map[well_id_src] = "Well_ID"
    if site_id_src: rename_map[site_id_src] = "Site_ID"
    rename_map[cat_col_src] = "Cat"
    if depth_col_src: rename_map[depth_col_src] = "DepthEnd"
    sub = sub.rename(columns=rename_map)
    wells_frames.append(sub)

if not wells_frames:
    raise ValueError(f"No suitable sheet in {wells_path} with an ID column and Cat column.")

wells = pd.concat(wells_frames, ignore_index=True)

# Normalize ID strings and depth
for col in ["Location_ID", "Well_ID", "Site_ID"]:
    if col in wells.columns:
        wells[col] = wells[col].astype(str).str.strip()
if "DepthEnd" in wells.columns:
    wells["DepthEnd"] = pd.to_numeric(wells["DepthEnd"], errors="coerce")
else:
    wells["DepthEnd"] = np.nan

# Choose a name to parse screens from (prefer Well_ID, else Location_ID, else Site_ID)
def name_for_parsing(row):
    for c in ["Well_ID", "Location_ID", "Site_ID"]:
        if c in wells.columns:
            v = row.get(c, None)
            if pd.notna(v) and str(v).strip() not in ("", "nan", "None"):
                return str(v).strip()
    return ""

# Parse base well and explicit screen number "S#"
def parse_base_screen(name: str):
    s = str(name).strip()
    m = re.search(r"(.*?)[\s_-]*s\s*(\d+)$", s, flags=re.IGNORECASE)
    if m:
        base = m.group(1).strip()
        try:
            scr = int(m.group(2))
        except Exception:
            scr = None
        return base, scr
    return s, None

wells["NameParse"] = wells.apply(name_for_parsing, axis=1)
wells["BaseWell"], wells["ScreenExplicit"] = zip(*wells["NameParse"].map(parse_base_screen))

# Depth-based rank within each base well: shallower DepthEnd => rank 1 (S1)
wells["RankByDepth"] = np.nan
for base_u, g in wells.groupby(wells["BaseWell"].str.upper(), dropna=False):
    if g["DepthEnd"].notna().any():
        wells.loc[g.index, "RankByDepth"] = g["DepthEnd"].rank(method="dense")  # ascending: S1, S2, ...
    else:
        if g["ScreenExplicit"].notna().any():
            wells.loc[g.index, "RankByDepth"] = g["ScreenExplicit"]
        else:
            wells.loc[g.index, "RankByDepth"] = 1.0 if len(g) == 1 else np.nan

# Exact maps
exact_map_loc  = {str(k).upper(): v for k, v in wells[["Location_ID", "Cat"]].itertuples(index=False)} if "Location_ID" in wells else {}
exact_map_well = {str(k).upper(): v for k, v in wells[["Well_ID", "Cat"]].itertuples(index=False)} if "Well_ID" in wells else {}

# Screen tuple map: (BaseWell, rank by depth) -> Cat
tuple_map = {}
for _, row in wells.iterrows():
    base_u = str(row["BaseWell"]).upper()
    key_scr = None
    if pd.notna(row["RankByDepth"]):
        key_scr = int(row["RankByDepth"])
    elif pd.notna(row["ScreenExplicit"]):
        key_scr = int(row["ScreenExplicit"])
    if key_scr is not None and base_u:
        tuple_map[(base_u, key_scr)] = row["Cat"]

# Singleton fallback: base well present exactly once
singleton_cat = (
    wells.groupby(wells["BaseWell"].str.upper(), dropna=False)["Cat"]
         .agg(lambda s: s.iloc[0] if len(s) == 1 else np.nan)
)

def normalize_key(s):
    return re.sub(r"\s+", " ", str(s or "")).strip().upper()

match_counts = {"exact_location": 0, "exact_well": 0, "screen_rank": 0, "singleton_base": 0}

def cat_for_location(loc: str):
    s = normalize_key(loc)
    if not s:
        return np.nan
    # Exact by Location_ID
    if s in exact_map_loc:
        match_counts["exact_location"] += 1
        return exact_map_loc[s]
    # Exact by Well_ID
    if s in exact_map_well:
        match_counts["exact_well"] += 1
        return exact_map_well[s]
    # Screen-based (e.g., 'R-70 S2')
    base, scr = parse_base_screen(s)
    base_u = normalize_key(base)
    if scr is not None:
        val = tuple_map.get((base_u, int(scr)))
        if val is not None:
            match_counts["screen_rank"] += 1
            return val
    # Singleton base fallback
    val = singleton_cat.get(base_u, np.nan)
    if pd.notna(val):
        match_counts["singleton_base"] += 1
        return val
    return np.nan

# Compute Cat values
cat_values = df[location_col].map(cat_for_location)

# Capture rows that originally had no Cat
unmatched_mask = cat_values.isna() | (cat_values.astype(str).str.strip() == "")
unmatched = df.loc[unmatched_mask, [location_col, date_col]].copy().sort_values([location_col, date_col])

# Dataset-specific rule: assign "Intermediate" to any remaining blanks
cat_values_filled = cat_values.copy()
cat_values_filled = cat_values_filled.fillna("Intermediate")
cat_values_filled = cat_values_filled.replace("", "Intermediate")

# Overwrite the Date column IN PLACE (same position) with 'Cat'
out = df.copy()
date_idx = out.columns.get_loc(date_col)
# Drop the original date column and insert 'Cat' at the same index
out = out.drop(columns=[date_col])
out.insert(date_idx, "Cat", cat_values_filled)

# Save outputs
out.to_csv(output_matrix, index=False, na_rep="")
if len(unmatched) > 0:
    unmatched.to_csv(unmatched_out, index=False)
else:
    pd.DataFrame(columns=[location_col, date_col]).to_csv(unmatched_out, index=False)

# Report
total = len(out)
matched = total - len(unmatched)
print(f"✅ Input matrix: {input_matrix}")
print(f"✅ Wells catalog: {wells_path}")
print(f"✅ Wrote matrix with Cat overwriting '{date_col}' at the same position: {output_matrix}")
print(f"✅ Unmatched report (pre-fill) written to: {unmatched_out} (count={len(unmatched)})")
print(f"Resolved columns -> Location: '{location_col}' | Date: '{date_col}' | Aluminum: '{al_col}'")
print(f"Cat column position: {out.columns.get_loc('Cat')} (should equal original Date position {date_idx})")
print(f"Match breakdown: {match_counts} | Matched (pre-fill) {matched}/{total} rows")

✅ Input matrix: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\CrWedge_AnalytesTruncated_MR.csv
✅ Wells catalog: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\WellsOfInterest.xls
✅ Wrote matrix with Cat overwriting 'Date Sampled' at the same position: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\CrWedge_AnalytesTruncated_MR_withCat.csv
✅ Unmatched report (pre-fill) written to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\CrWedge_Analytes_UnmatchedToCat.csv (count=11)
Resolved columns -> Location: 'Location' | Date: 'Date Sampled' | Aluminum: 'Aluminum'
Cat column position: 1 (should equal original Date position 1)
Match breakdown: {'exact_location': 33, 'exact_well': 0, 'screen_rank': 40, 'singleton_base': 4} | Matched (pre-fill) 76/87 rows


In [5]:
#!/usr/bin/env python3
"""
Build a simple profile for numeric analytes in AllWells_V3_nInt.

Fixes applied:
 - Corrected profile_out path construction (previous code used an invalid `base_dir / + "/..."`).
 - Ensure output directory is created before writing.
 - Made resolve_col robust to capitalization/whitespace and return the original column name.
 - Guard against missing optional Cat column.
 - Ensure id_like excludes None values.
 - Simplified and corrected zero count calculation.
 - Improved some small robustness checks and clearer error messages.

Run from a Jupyter cell with:
    %run Intellus_profile_builder.py
"""

from pathlib import Path
import pandas as pd
import numpy as np

# -------- Paths (edit if needed) --------
base_dir = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items")
input_path = base_dir / "AllWells_V3_nInt.csv"
profile_out = base_dir / "results_3.1" / "AllWells_V3_nInt_pca_profile.csv"
# ---------------------------------------

# Load input (support csv or xlsx)
if not input_path.exists():
    raise FileNotFoundError(f"Input not found: {input_path}")
if input_path.suffix.lower() == ".xlsx":
    df = pd.read_excel(input_path)
else:
    df = pd.read_csv(input_path)

# Resolve identifiers (case-insensitive, whitespace-insensitive)
lower_map = {c.lower().strip(): c for c in df.columns}
def resolve_col(options, label, required=True):
    for o in options:
        key = o.lower().strip()
        if key in lower_map:
            return lower_map[key]
    if required:
        raise ValueError(f"Missing required column for {label}. Looked for {options}. Found: {list(df.columns)}")
    return None

location_col = resolve_col(["location", "location id", "location_id"], "Location")
cat_col      = resolve_col(["cat"], "Cat", required=False)

# Numeric analyte columns: exclude Location and Cat only (if cat_col not found, ignore)
id_like = {location_col, cat_col} - {None}
candidate_cols = [c for c in df.columns if c not in id_like]

# Coerce candidates to numeric when possible (>=70% numeric or column dtype numeric)
numeric_cols = []
for c in candidate_cols:
    s_num = pd.to_numeric(df[c], errors="coerce")
    frac_numeric = s_num.notna().mean() if len(s_num) else 0.0
    if frac_numeric >= 0.7 or pd.api.types.is_numeric_dtype(df[c]):
        numeric_cols.append(c)
if not numeric_cols:
    raise ValueError("No numeric analyte columns detected based on the current heuristics.")

# Ensure output directory exists
profile_out.parent.mkdir(parents=True, exist_ok=True)

# Profile each numeric analyte
rows = []
for c in numeric_cols:
    s = pd.to_numeric(df[c], errors="coerce")
    n = len(s)
    nmiss = int(s.isna().sum())
    miss_pct = 100.0 * nmiss / n if n else np.nan
    s_valid = s.dropna()
    zero_ct = int((s_valid == 0).sum())
    nonpos_ct = int((s_valid <= 0).sum())
    uniq = int(s_valid.nunique(dropna=True))
    const = (uniq <= 1)
    vmin = float(np.nanmin(s_valid)) if len(s_valid) else np.nan
    vmax = float(np.nanmax(s_valid)) if len(s_valid) else np.nan
    p1 = float(np.nanpercentile(s_valid, 1)) if len(s_valid) else np.nan
    p50 = float(np.nanpercentile(s_valid, 50)) if len(s_valid) else np.nan
    p99 = float(np.nanpercentile(s_valid, 99)) if len(s_valid) else np.nan
    skew = float(pd.Series(s_valid).skew()) if len(s_valid) else np.nan

    if len(s_valid) == 0 or const:
        sugg = "drop (no variance or no data)"
    else:
        skew_val = skew if np.isfinite(skew) else 0.0
        if nonpos_ct == 0 and skew_val > 1.0:
            sugg = "log10"
        elif skew_val > 1.0:
            sugg = "yeo-johnson"
        else:
            sugg = "none"

    rows.append({
        "parameter": c,
        "n": n,
        "missing_count": nmiss,
        "missing_pct": round(miss_pct, 2),
        "unique_non_nan": uniq,
        "zero_count": zero_ct,
        "nonpositive_count": nonpos_ct,
        "min": vmin,
        "p01": p1,
        "p50": p50,
        "p99": p99,
        "max": vmax,
        "skew": skew,
        "suggested_transform": sugg,
    })

profile = pd.DataFrame(rows).sort_values(["suggested_transform", "missing_pct", "parameter"])
profile.to_csv(profile_out, index=False)

print(f"✅ Profile written: {profile_out}")
print(f"- Numeric analytes profiled: {len(numeric_cols)}")
print(f"- High missing (>20%): {profile[profile['missing_pct'] > 20].shape[0]}")
print(f"- Constant/no-variance: {profile[profile['suggested_transform'].str.startswith('drop')].shape[0]}")
print(f"- Log10 candidates: {profile[profile['suggested_transform']=='log10'].shape[0]}")
print(f"- Yeo-Johnson candidates: {profile[profile['suggested_transform']=='yeo-johnson'].shape[0]}")

✅ Profile written: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\AllWells_V3_nInt_pca_profile.csv
- Numeric analytes profiled: 22
- High missing (>20%): 0
- Constant/no-variance: 0
- Log10 candidates: 22
- Yeo-Johnson candidates: 0


In [8]:
#!/usr/bin/env python3
r"""
PCA scores creator + plotter

This script will:
 - Use results_3.1 under V 2.1 items as the results folder.
 - If PCA outputs (pca_scores.csv, pca_rotation.csv, pca_explained_variance.csv) exist, it will load them.
 - If not, it will build PCA from a numeric matrix source (default: AllWells_V3_nInt.csv),
   save scores, rotation (loadings), and explained-variance CSVs into results_3.1.
 - Print percent variance explained for the first three PCs.
 - Save a PNG bar plot of percent explained for all PCs.
 - Save a labeled PC1 vs PC2 scatter (with convex hull labels) into results_3.1 and export hull vertices.

Notes:
 - PCA building uses sklearn (SimpleImputer + StandardScaler + PCA). Install scikit-learn if missing.
 - Adjust numeric-detection heuristics (70% numeric) if you want different behavior.
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import patheffects as mpe

# Optional label de-overlap
try:
    from adjustText import adjust_text
    HAVE_ADJUSTTEXT = True
except Exception:
    HAVE_ADJUSTTEXT = False

# PCA dependencies
try:
    from sklearn.decomposition import PCA
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
except Exception as e:
    raise ImportError("scikit-learn is required. Install with: pip install scikit-learn") from e

# ---------------- Paths ----------------
BASE_DIR = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items")
RESULTS_DIR = BASE_DIR / "results_3.1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SCORES_PATH = RESULTS_DIR / "pca_scores.csv"
ROTATION_PATH = RESULTS_DIR / "pca_rotation.csv"
EXPLAINED_CSV = RESULTS_DIR / "pca_explained_variance.csv"
EXPLAINED_PNG = RESULTS_DIR / "pca_explained_variance.png"
SCATTER_PNG = RESULTS_DIR / "pca_pc1_pc2_scatter_labeled_deoverlap.png"
HULL_CSV = RESULTS_DIR / "pca_convex_hull_vertices.csv"

# Default matrix source to build PCA from if scores not present
MATRIX_SOURCE = BASE_DIR / "AllWells_V3_nInt.csv"
# ---------------------------------------

def resolve_col_in_df(df, candidates, required=True):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in lower_map:
            return lower_map[key]
    if required:
        raise ValueError(f"Missing required column among: {candidates}. Available cols: {list(df.columns)}")
    return None

def detect_numeric_features(df, exclude_cols):
    id_like = set(exclude_cols) - {None}
    candidate_cols = [c for c in df.columns if c not in id_like]
    numeric_cols = []
    for c in candidate_cols:
        s_num = pd.to_numeric(df[c], errors="coerce")
        frac_numeric = s_num.notna().mean() if len(s_num) else 0.0
        if frac_numeric >= 0.7 or pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)
    return numeric_cols

def build_and_save_pca(df_matrix, matrix_path, numeric_cols, location_col=None, cat_col=None, n_components=None):
    # Extract numeric feature matrix
    X = df_matrix[numeric_cols].copy()
    X_num = X.apply(pd.to_numeric, errors="coerce")

    # Impute missing -> column mean
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(X_num.values)

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # Determine n_components
    if n_components is None:
        n_components = min(X_scaled.shape[0], X_scaled.shape[1])

    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X_scaled)

    # Build scores DataFrame
    pc_cols = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
    scores_df = pd.DataFrame(scores, columns=pc_cols, index=df_matrix.index)

    # Insert identifier columns if available
    if location_col and location_col in df_matrix.columns:
        scores_df.insert(0, location_col, df_matrix[location_col].astype(str).values)
    else:
        scores_df.insert(0, "Location", [""] * len(scores_df))

    if cat_col and cat_col in df_matrix.columns:
        scores_df.insert(1, cat_col, df_matrix[cat_col].astype(str).values)
    else:
        scores_df.insert(1, "Cat", [""] * len(scores_df))

    # Save scores
    scores_df.to_csv(SCORES_PATH, index=False, float_format="%.8g")

    # Rotation/loadings: features x PCs
    loadings = pd.DataFrame(pca.components_.T, index=numeric_cols, columns=pc_cols)
    loadings.to_csv(ROTATION_PATH, index=True, float_format="%.8g")

    # Explained variance ratios
    pct = 100.0 * pca.explained_variance_ratio_
    explained_df = pd.DataFrame({"PC": pc_cols, "percent_explained": pct})
    explained_df.to_csv(EXPLAINED_CSV, index=False, float_format="%.8g")

    return scores_df, loadings, pct

def detect_pc_columns(columns):
    pc_list = []
    pat = re.compile(r'(?:^|[^a-z0-9])pc[^\d]*(\d+)', flags=re.IGNORECASE)
    pat2 = re.compile(r'(?:^|[^a-z0-9])pca[^\d]*(\d+)', flags=re.IGNORECASE)
    for col in columns:
        col_low = str(col).lower()
        m = pat.search(col_low) or pat2.search(col_low)
        if m:
            idx = int(m.group(1))
            pc_list.append((idx, col))
            continue
        m2 = re.search(r'\bcomponent[^\d]*(\d+)\b', col_low)
        if m2:
            pc_list.append((int(m2.group(1)), col))
            continue
        m3 = re.search(r'\bpc_(\d+)\b', col_low)
        if m3:
            pc_list.append((int(m3.group(1)), col))
            continue
    pc_list_sorted = [col for idx, col in sorted(pc_list, key=lambda x: x[0])]
    return pc_list_sorted

# ---------------- Main ----------------

# Try to load existing PCA outputs; otherwise build
if SCORES_PATH.exists() and ROTATION_PATH.exists() and EXPLAINED_CSV.exists():
    df_scores = pd.read_csv(SCORES_PATH)
    df_rotation = pd.read_csv(ROTATION_PATH, index_col=0)
    explained_df = pd.read_csv(EXPLAINED_CSV)
    pct_explained = explained_df["percent_explained"].values
    pc_cols_all = detect_pc_columns(df_scores.columns)
    if not pc_cols_all:
        # fallback to columns matching PC\d+
        fallback = [c for c in df_scores.columns if re.fullmatch(r'pc\d+', str(c).strip().lower())]
        pc_cols_all = sorted(fallback, key=lambda c: int(re.search(r'(\d+)', c).group(1))) if fallback else []
else:
    # Build PCA from source matrix
    if not MATRIX_SOURCE.exists():
        raise FileNotFoundError(
            f"PCA scores not found in {RESULTS_DIR} and matrix source not found: {MATRIX_SOURCE}"
        )

    df_mat = pd.read_csv(MATRIX_SOURCE, keep_default_na=True)

    # Detect identifier columns
    location_col = resolve_col_in_df(df_mat, ["location", "location id", "location_id"], required=False)
    cat_col = resolve_col_in_df(df_mat, ["cat", "category"], required=False)

    numeric_cols = detect_numeric_features(df_mat, exclude_cols=[location_col, cat_col])
    if not numeric_cols:
        raise ValueError(f"No numeric features detected in matrix source: {MATRIX_SOURCE}")

    # Build PCA and save outputs
    df_scores, df_rotation, pct_explained = build_and_save_pca(df_mat, MATRIX_SOURCE, numeric_cols, location_col, cat_col)
    pc_cols_all = [f"PC{i}" for i in range(1, len(pct_explained) + 1)]

# Print percent explained for first 3 PCs
print("Percent variance explained (first 3 PCs):")
for i in range(min(3, len(pct_explained))):
    print(f"  PC{i+1}: {pct_explained[i]:.2f}%")

# Save explained variance bar plot (PNG)
plt.figure(figsize=(10, 5))
pc_labels = [f"PC{i+1}" for i in range(len(pct_explained))]
sns.barplot(x=pc_labels, y=pct_explained, color="C0")
plt.xticks(rotation=45)
plt.ylabel("% Variance Explained")
plt.xlabel("Principal Component")
plt.title("PCA: Percent Variance Explained by PC")
for i, v in enumerate(pct_explained):
    plt.text(i, v + 0.3, f"{v:.2f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(EXPLAINED_PNG, dpi=300, bbox_inches="tight")
plt.close()
print(f"✅ Explained-variance figure saved: {EXPLAINED_PNG}")

# Prepare scatter data (use df_scores variable from build or load)
if 'df_scores' not in locals():
    df_scores = pd.read_csv(SCORES_PATH)

# Resolve identifiers and PC1/PC2 names in df_scores
location_col = resolve_col_in_df(df_scores, ["location", "location id", "location_id", "site"], required=True)
cat_col = resolve_col_in_df(df_scores, ["cat", "category"], required=True)

pc_cols_detected = detect_pc_columns(df_scores.columns)
if not pc_cols_detected:
    pc_cols_detected = [c for c in df_scores.columns if re.search(r'pc\d+', str(c).strip().lower())]
if not pc_cols_detected:
    raise ValueError("No PC columns detected in scores dataframe.")

pc1_col = pc_cols_detected[0]
pc2_col = pc_cols_detected[1] if len(pc_cols_detected) > 1 else pc_cols_detected[0]

plot_df = df_scores[[location_col, cat_col, pc1_col, pc2_col]].copy().dropna(subset=[pc1_col, pc2_col])
plot_df[pc1_col] = pd.to_numeric(plot_df[pc1_col], errors="coerce")
plot_df[pc2_col] = pd.to_numeric(plot_df[pc2_col], errors="coerce")

# Anthropogenic mask (case-insensitive)
anthro_mask = plot_df[cat_col].astype(str).str.strip().str.lower().eq("anthropogenic")

# Convex hull
def convex_hull_indices(xy: np.ndarray):
    uniq_pts = np.unique(xy, axis=0)
    if len(uniq_pts) < 3:
        return np.array([], dtype=int)
    order = np.argsort(uniq_pts[:, 0], kind="mergesort")
    pts = uniq_pts[order]
    idxs = np.arange(len(uniq_pts))[order]
    def cross(o, a, b):
        return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    lower, lower_idx = [], []
    for i, p in enumerate(pts):
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop(); lower_idx.pop()
        lower.append(p); lower_idx.append(idxs[i])
    upper, upper_idx = [], []
    for i, p in enumerate(pts[::-1]):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop(); upper_idx.pop()
        upper.append(p); upper_idx.append(idxs[len(pts)-1-i])
    hull_unique_idx = np.array(lower_idx[:-1] + upper_idx[:-1], dtype=int)
    hull_points = uniq_pts[hull_unique_idx]
    original_indices = []
    for hp in hull_points:
        matches = np.where((xy == hp).all(axis=1))[0]
        original_indices.append(matches[0])
    return np.array(original_indices, dtype=int)

coords = plot_df[[pc1_col, pc2_col]].to_numpy()
try:
    from scipy.spatial import ConvexHull
    if coords.shape[0] >= 3 and np.unique(coords, axis=0).shape[0] >= 3:
        hull = ConvexHull(coords)
        hull_idx = hull.vertices
    else:
        hull_idx = np.array([], dtype=int)
except Exception:
    hull_idx = convex_hull_indices(coords)

hull_mask = np.zeros(len(plot_df), dtype=bool)
if hull_idx.size > 0:
    hull_mask[hull_idx] = True

# Scatter with labels
plt.figure(figsize=(10.5, 8.5))
ax = sns.scatterplot(
    data=plot_df,
    x=pc1_col, y=pc2_col,
    hue=cat_col, palette="tab10",
    s=70, edgecolor="k", linewidth=0.6, alpha=0.95
)

def outline(text_obj, lw=3):
    text_obj.set_path_effects([mpe.Stroke(linewidth=lw, foreground="white"), mpe.Normal()])

texts = []
for i in np.where(hull_mask)[0]:
    row = plot_df.iloc[i]
    t = ax.text(row[pc1_col] + 0.01, row[pc2_col] + 0.01, f"{row[location_col]}\n{row[cat_col]}", fontsize=9.5, weight="bold", zorder=6)
    outline(t, lw=3)
    texts.append(t)

for i in np.where(anthro_mask)[0]:
    if hull_mask[i]:
        continue
    row = plot_df.iloc[i]
    t = ax.text(row[pc1_col] + 0.01, row[pc2_col] - 0.01, f"{row[location_col]}", fontsize=9, weight="bold", zorder=6)
    outline(t, lw=3)
    texts.append(t)

if HAVE_ADJUSTTEXT and texts:
    adjust_text(
        texts,
        x=plot_df[pc1_col].values, y=plot_df[pc2_col].values,
        expand_points=(1.2, 1.2),
        force_points=(0.3, 0.3),
        force_text=(0.4, 0.4),
        only_move={"points": "none", "text": "xy"},
        autoalign="xy",
        lim=200,
        arrowprops=None
    )
else:
    for j, t in enumerate(texts):
        dx = (j % 5) * 0.01
        dy = ((j // 5) % 5) * 0.01
        t.set_position((t.get_position()[0] + dx, t.get_position()[1] + dy))

leg = ax.legend(title="Cat", loc="lower right", frameon=True, framealpha=0.9)
if leg:
    leg.set_zorder(1)

ax.margins(x=0.08, y=0.08)
plt.title("PCA: PC1 vs PC2 (Hull: Location+Cat; Anthropogenic: Location)")
plt.xlabel(pc1_col); plt.ylabel(pc2_col)
plt.tight_layout()
plt.savefig(SCATTER_PNG, dpi=240, bbox_inches="tight")
plt.close()

# Save hull vertices table
hull_table = plot_df.iloc[hull_idx][[location_col, cat_col, pc1_col, pc2_col]].copy() if hull_idx.size > 0 else pd.DataFrame(columns=[location_col, cat_col, pc1_col, pc2_col])
hull_table.to_csv(HULL_CSV, index=False)

print(f"✅ PCA scores/rotation/explained are in: {RESULTS_DIR}")
print(f"✅ Scatter saved: {SCATTER_PNG}")
print(f"✅ Explained-variance saved: {EXPLAINED_PNG}")
print(f"✅ Convex hull vertices saved: {HULL_CSV} (n={len(hull_table)})")

Percent variance explained (first 3 PCs):
  PC1: 36.25%
  PC2: 16.28%
  PC3: 8.60%
✅ Explained-variance figure saved: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_explained_variance.png
✅ PCA scores/rotation/explained are in: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1
✅ Scatter saved: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_pc1_pc2_scatter_labeled_deoverlap.png
✅ Explained-variance saved: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_explained_variance.png
✅ Convex hull vertices saved: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_convex_hull_vertices.csv (n=10)


In [2]:
#!/usr/bin/env python3
r"""
Interactive 3D PCA scatter with selected labels and 3D convex-hull export.

Behavior:
 - Finds pca_scores.csv (preferring results_3.1 under V 2.1 items).
 - Ensures PC1/PC2/PC3 numeric columns (fills PC3 with zeros if missing).
 - Computes the convex hull in 3-space (PC1,PC2,PC3) and writes hull vertices to CSV.
 - Labels (projected) on the 3D scatter:
     * all 3D hull vertices (Location + Cat),
     * all rows with Cat == "Anthropogenic" (Location),
     * explicit locations: ["CrPZ-3", "R-44 S1", "R-45 S1", "R-42"] (case-insensitive, non-alphanumeric ignored)
 - Removes legend entries whose label includes the string "Aa" (font-sample items).
 - Writes interactive HTML only (no PNG snapshot).
 - Writes pca_3d_hull_vertices.csv with the 3D hull vertices.

Requirements:
    pip install pandas scipy plotly

Notes:
 - PNG snapshot (kaleido) has been removed per request so no kaleido dependency or PNG write occurs.
"""

from pathlib import Path
import re
import sys
import gc
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# ---------- Robust path resolution (prefer results_3.1 root) ----------
DEFAULT_ROOT = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items")
PREFERRED_SUBDIR = "results_3.1"

def find_pca_scores(root: Path = DEFAULT_ROOT, preferred_subdir: str = PREFERRED_SUBDIR):
    cand = root / preferred_subdir / "pca_scores.csv"
    if cand.exists():
        return cand
    cand2 = root / preferred_subdir / "PCA_Step1" / "pca_scores.csv"
    if cand2.exists():
        return cand2
    cand3 = root / "PCA_Step1" / "pca_scores.csv"
    if cand3.exists():
        return cand3
    found = list(root.rglob("pca_scores.csv"))
    if not found:
        cwd_found = list(Path.cwd().rglob("pca_scores.csv"))
        if cwd_found:
            found = cwd_found
    if not found:
        raise FileNotFoundError(f"No pca_scores.csv found under {root} or cwd {Path.cwd()}.")
    for f in found:
        if preferred_subdir.lower() in str(f).lower():
            return f
    # fallback to first found (and print list for debugging)
    print("Multiple/alternate pca_scores.csv candidates found; using the first one. Full list:")
    for f in found:
        print(" -", f)
    return found[0]

SCORES_PATH = find_pca_scores()
BASE_DIR = SCORES_PATH.parent   # directory containing pca_scores.csv (PCA_v2 root or results_3.1)
OUT_DIR = BASE_DIR
OUT_HTML = OUT_DIR / "pca_3d_interactive.html"
HULL_CSV_3D = OUT_DIR / "pca_3d_hull_vertices.csv"

print(f"Using PCA scores at: {SCORES_PATH}")
print(f"Outputs will be written to: {OUT_DIR}")

# ---- Load data ----
df = pd.read_csv(SCORES_PATH)

# Resolve columns (case-insensitive helpers)
def resolve_col(df, candidates):
    low_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in low_map:
            return low_map[key]
    raise KeyError(f"Could not resolve any of {candidates} in columns: {list(df.columns)[:20]}")

location_col = resolve_col(df, ["location", "location id", "location_id"])
cat_col = resolve_col(df, ["cat", "category"])
pc1_col = resolve_col(df, ["pc1", "PC1", "PC 1", "pc 1"])
pc2_col = resolve_col(df, ["pc2", "PC2", "PC 2", "pc 2"])
# PC3 optional: if missing, create PC3 filled with zeros so we can still view in 3D
try:
    pc3_col = resolve_col(df, ["pc3", "PC3", "PC 3", "pc 3"])
    has_pc3 = True
except KeyError:
    df["PC3"] = 0.0
    pc3_col = "PC3"
    has_pc3 = False
    print("PC3 not found; using zeros for PC3 for a flat 3D view.")

# coerce numeric
for c in [pc1_col, pc2_col, pc3_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# drop rows missing PC1/PC2 (PC3 can be NaN only if we created it)
df_clean = df.dropna(subset=[pc1_col, pc2_col]).copy()
if df_clean.empty:
    raise ValueError("No valid PC1/PC2 data found after coercion to numeric.")

# Normalized keys for matching explicit labels and R-42:
def norm_key(s: str):
    # remove non-alphanumeric, lower-case
    return re.sub(r'[^a-z0-9]', '', str(s or "").lower())

df_clean["_loc_norm_alnum"] = df_clean[location_col].astype(str).map(norm_key)

# Prepare coords for 3D hull (PC1, PC2, PC3)
coords3 = df_clean[[pc1_col, pc2_col, pc3_col]].to_numpy()

# Compute 3D convex hull vertices (indices into coords3 array) and map to original df_clean indices
hull3_idx_local = np.array([], dtype=int)
have_scipy = False
try:
    from scipy.spatial import ConvexHull
    have_scipy = True
except Exception:
    have_scipy = False

if coords3.shape[0] >= 4 and np.unique(coords3, axis=0).shape[0] >= 4 and have_scipy:
    try:
        hull3 = ConvexHull(coords3)
        hull3_idx_local = np.array(hull3.vertices, dtype=int)
    except Exception as e:
        print("scipy ConvexHull failed:", e)
        hull3_idx_local = np.array([], dtype=int)
else:
    # not enough unique points or scipy missing -> no 3D hull
    hull3_idx_local = np.array([], dtype=int)

# Map local indices to the df_clean global indices
if hull3_idx_local.size > 0:
    df_clean_local_index_array = df_clean.index.to_numpy()
    hull3_idx_global = df_clean_local_index_array[hull3_idx_local]
else:
    hull3_idx_global = np.array([], dtype=int)

# Write 3D hull vertices CSV
if hull3_idx_global.size > 0:
    hull3_table = df_clean.loc[hull3_idx_global, [location_col, cat_col, pc1_col, pc2_col, pc3_col]].copy()
else:
    hull3_table = pd.DataFrame(columns=[location_col, cat_col, pc1_col, pc2_col, pc3_col])
hull3_table.to_csv(HULL_CSV_3D, index=False)
print(f"3D convex hull vertices written to: {HULL_CSV_3D} (n={len(hull3_table)})")

# Build base 3D scatter with Plotly Express for categorical coloring
fig_base = px.scatter_3d(
    df_clean,
    x=pc1_col, y=pc2_col, z=pc3_col,
    color=cat_col,
    hover_name=location_col,
    opacity=0.9,
    title="PCA 3D: PC1 vs PC2 vs PC3"
)
fig = go.Figure(fig_base)

# Add 3D hull vertex labels (Location + Cat) projected into 3D points
if hull3_idx_local.size > 0:
    hull3_df = df_clean.iloc[hull3_idx_local].copy().reset_index(drop=True)
    hull3_text = hull3_df[location_col].astype(str) + "<br>" + hull3_df[cat_col].astype(str)
    fig.add_trace(
        go.Scatter3d(
            x=hull3_df[pc1_col],
            y=hull3_df[pc2_col],
            z=hull3_df[pc3_col],
            mode="text",
            text=hull3_text,
            textposition="top center",
            textfont=dict(size=11, color="black", family="Arial"),
            hoverinfo="text",
            name="Hull vertices (3D)",
            showlegend=False,   # labels speak for themselves
            marker=dict(size=1),
            opacity=1.0,
        )
    )

# Add anthropogenic labels (Location), skipping ones already labeled as hull
anthro_mask_local = df_clean[cat_col].astype(str).str.strip().str.lower() == "anthropogenic"
if anthro_mask_local.any():
    # exclude hull-local positions
    anthro_df = df_clean[anthro_mask_local].copy()
    if hull3_idx_local.size > 0:
        anthro_df = anthro_df.loc[~anthro_df.index.isin(hull3_idx_global)]
    if not anthro_df.empty:
        fig.add_trace(
            go.Scatter3d(
                x=anthro_df[pc1_col],
                y=anthro_df[pc2_col],
                z=anthro_df[pc3_col],
                mode="text",
                text=anthro_df[location_col].astype(str),
                textposition="middle right",
                textfont=dict(size=10, color="black", family="Arial"),
                hoverinfo="text",
                name="Anthropogenic (Location)",
                showlegend=False,  # labels speak for themselves
                marker=dict(size=1),
                opacity=1.0,
            )
        )

# Explicit labels requested by user (include R-42 as well)
EXTRA_LABELS = ["CrPZ-3", "R-44 S1", "R-45 S1", "R-42"]
# create normalized set
norm_targets = {norm_key(t): t for t in EXTRA_LABELS}
found_local_idx = []
for idx_local, loc_norm in enumerate(df_clean["_loc_norm_alnum"].values):
    if loc_norm in norm_targets:
        found_local_idx.append(idx_local)

if found_local_idx:
    extra_df = df_clean.iloc[found_local_idx].copy().reset_index(drop=True)
    extra_text = extra_df[location_col].astype(str)
    fig.add_trace(
        go.Scatter3d(
            x=extra_df[pc1_col],
            y=extra_df[pc2_col],
            z=extra_df[pc3_col],
            mode="text",
            text=extra_text,
            textposition="bottom center",
            textfont=dict(size=12, color="black", family="Arial"),
            hoverinfo="text",
            name="Explicit labels",
            showlegend=False,
            marker=dict(size=1),
            opacity=1.0,
        )
    )
    print(f"Added explicit labels for: {', '.join(extra_df[location_col].astype(str).tolist())}")
else:
    print("No explicit label matches found for the requested list. Check exact Location names in pca_scores.csv.")

# Make sure R-42 is included: if no explicit match above, try a substring search in normalized keys
if not any(norm_key(t) in df_clean["_loc_norm_alnum"].values for t in ["r42"]):
    # search by substring match on raw location strings
    r42_mask = df_clean[location_col].astype(str).str.replace(r'\s+', '', regex=True).str.upper().str.contains("R42")
    if r42_mask.any():
        r42_df = df_clean.loc[r42_mask].copy().reset_index(drop=True)
        # add labels for these if not already included
        fig.add_trace(
            go.Scatter3d(
                x=r42_df[pc1_col],
                y=r42_df[pc2_col],
                z=r42_df[pc3_col],
                mode="text",
                text=r42_df[location_col].astype(str),
                textposition="middle left",
                textfont=dict(size=12, color="black", family="Arial"),
                hoverinfo="text",
                name="R-42 (match)",
                showlegend=False,
                marker=dict(size=1),
                opacity=1.0,
            )
        )
        print(f"Added R-42 labels for: {', '.join(r42_df[location_col].astype(str).tolist())}")

# Legend cleanup: hide traces whose legend label contains "Aa" (font-sample artifacts).
# Also hide any internal annotation traces we explicitly set showlegend=False for already.
for tr in fig.data:
    try:
        if tr.name and "Aa" in str(tr.name):
            tr.showlegend = False
    except Exception:
        pass

# Layout tweaks: move legend outside and centered vertically; keep traceorder normal
fig.update_layout(
    legend=dict(
        title="Cat",
        x=1.02,
        y=0.5,
        xanchor="left",
        yanchor="middle",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="rgba(200,200,200,0.6)",
        borderwidth=0.6,
        traceorder="normal",
    ),
    margin=dict(l=60, r=240, t=80, b=60),
    scene=dict(
        xaxis_title=pc1_col,
        yaxis_title=pc2_col,
        zaxis_title=pc3_col,
    ),
    title=dict(x=0.5),
)

# Save interactive HTML (the single desired output)
fig.write_html(str(OUT_HTML), include_plotlyjs="cdn")
print(f"Wrote interactive 3D HTML to: {OUT_HTML}")

# Save 3D hull vertices CSV already done above; report
total_points = len(df_clean)
anthro_count = int(anthro_mask_local.sum())
hull3_count = len(hull3_table)

print("\nSummary:")
print(f" - total points plotted: {total_points}")
print(f" - anthropogenic labeled points: {anthro_count}")
print(f" - 3D convex hull vertices saved: {hull3_count}")
if not has_pc3:
    print(" - NOTE: PC3 was not present in input; plotted PC3 as zeros (flat).")

# Notebook-star explanation
print("\nNote: If you see a running-star [*] next to the notebook cell, it means the kernel is still busy.")
print("If the script finished producing files but the cell still shows a star, interrupt the cell/kernel.")
print("A continuously busy kernel can block subsequent calculations; stop/interrupt it if it doesn't finish.")

# ---------------- Safe cleanup and graceful exit ----------------
# Close/cleanup large objects and exit so notebook cells finish without lingering busy state.
try:
    # Delete large objects we created
    for name in (
        "fig", "fig_base", "hull3", "hull3_df", "hull3_table", "hull3_idx_local",
        "hull3_idx_global", "coords3", "df_clean", "df", "df_clean_local_index_array",
        "r42_df", "extra_df", "anthro_df",
    ):
        if name in globals():
            try:
                del globals()[name]
            except Exception:
                pass

    # Close matplotlib if any (defensive)
    try:
        import matplotlib.pyplot as _plt
        _plt.close("all")
    except Exception:
        pass

    gc.collect()
except Exception:
    pass

# Exit the script cleanly. In a notebook this will end the cell execution.
try:
    sys.exit(0)
except SystemExit:
    # ensure termination if running as a script
    raise

Using PCA scores at: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_scores.csv
Outputs will be written to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1
3D convex hull vertices written to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_3d_hull_vertices.csv (n=18)
Added explicit labels for: CRPZ-3, R-42, R-44 S1, R-45 S1
Wrote interactive 3D HTML to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_3d_interactive.html

Summary:
 - total points plotted: 85
 - anthropogenic labeled points: 7
 - 3D convex hull vertices saved: 18

Note: If you see a running-star [*] next to the notebook cell, it means the kernel is still busy.
If the script finished producing files but the cell still shows a star, interrupt the cell/kernel.
A continuously busy kernel can block subsequent calculations; stop/interrupt it if it doesn't finish.


SystemExit: 0

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning:

To exit: use 'exit', 'quit', or Ctrl-D.



In [4]:
#!/usr/bin/env python3
"""
Fast plotting of PCA loadings for all variables (max -> min for each PC),
with consistent colors per parameter across panels, variable names shown
on every vertical axis, and no legend.

This script will:
 - Look for pca_loadings.csv in <DEFAULT_ROOT>/results_3.1/.
 - If not found, try to load an existing pca_rotation.csv and write it to pca_loadings.csv.
 - If neither exists, build PCA from the numeric matrix (AllWells_V3_nInt.csv),
   save loadings to pca_loadings.csv (and also pca_rotation.csv, pca_scores.csv, pca_explained_variance.csv)
   in results_3.1, then continue to plotting.
 - Produce:
   - pca_loadings_all_pcs_labeled_big.png
   - pca_loadings_all_pcs_labeled_big.pdf
   - pca_loadings_all_ordered.csv
"""

from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import gc
import sys

# PCA building dependencies
try:
    from sklearn.decomposition import PCA
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
except Exception as e:
    raise ImportError("scikit-learn is required to build loadings. Install with `pip install scikit-learn`.") from e

# ---- Configuration ----
DEFAULT_ROOT = Path(r"C:/Users/kylian.robinson/MapRebuildItems/V 2.1 items")
PCA_DIR = DEFAULT_ROOT / "results_3.1"
PCA_DIR.mkdir(parents=True, exist_ok=True)

LOADINGS_FILE = PCA_DIR / "pca_loadings.csv"
ROTATION_FILE = PCA_DIR / "pca_rotation.csv"   # alternate name used elsewhere
SCORES_FILE = PCA_DIR / "pca_scores.csv"
EXPLAINED_FILE = PCA_DIR / "pca_explained_variance.csv"

OUT_DIR = PCA_DIR
OUT_PNG = OUT_DIR / "pca_loadings_all_pcs_labeled_big.png"
OUT_PDF = OUT_DIR / "pca_loadings_all_pcs_labeled_big.pdf"
OUT_CSV = OUT_DIR / "pca_loadings_all_ordered.csv"

PC_NAMES_PREFERRED = ["PC1", "PC2", "PC3"]   # case-insensitive resolve

# Increased size and resolution for screenshot / document use
FIGSIZE = (30, 14)   # very wide
DPI = 300            # high resolution for crisp screenshots
FONT_SIZE = 11       # slightly larger font for readability in documents

# Source matrix if we need to build PCA
MATRIX_SOURCE = DEFAULT_ROOT / "AllWells_V3_nInt.csv"

# ---- Helpers ----
def detect_numeric_features(df, exclude_cols):
    id_like = set(exclude_cols) - {None}
    candidate_cols = [c for c in df.columns if c not in id_like]
    numeric_cols = []
    for c in candidate_cols:
        s_num = pd.to_numeric(df[c], errors="coerce")
        frac_numeric = s_num.notna().mean() if len(s_num) else 0.0
        if frac_numeric >= 0.7 or pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)
    return numeric_cols

def resolve_pc_names_from_df(df, preferred_list):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    def resolve(name):
        key = name.lower().strip()
        if key in cols_lower:
            return cols_lower[key]
        for c in cols_lower:
            if c.replace(" ", "").replace("_", "") == key.replace(" ", "").replace("_", ""):
                return cols_lower[c]
        return None
    resolved = [resolve(p) for p in preferred_list]
    return resolved

# ---- Ensure loadings exist (or build them) ----
t0 = time.time()

if not LOADINGS_FILE.exists():
    # Try to recover from alternate rotation file
    if ROTATION_FILE.exists():
        rot_df = pd.read_csv(ROTATION_FILE, index_col=0)
        rot_df.to_csv(LOADINGS_FILE)
        print(f"Recovered loadings from {ROTATION_FILE} -> wrote {LOADINGS_FILE}.")
    else:
        # Need to build PCA from matrix source
        if not MATRIX_SOURCE.exists():
            raise FileNotFoundError(
                f"No loadings found ({LOADINGS_FILE}) and no rotation file ({ROTATION_FILE}).\n"
                f"Also matrix source not found: {MATRIX_SOURCE}\n"
                "Place pca_loadings.csv or pca_rotation.csv in results_3.1, or provide the matrix file."
            )

        print(f"No loadings present. Building PCA from matrix source: {MATRIX_SOURCE}")
        df_mat = pd.read_csv(MATRIX_SOURCE, keep_default_na=True)

        # identify id columns (optional)
        lower_map = {c.lower().strip(): c for c in df_mat.columns}
        def resolve_col(options):
            for o in options:
                key = o.lower().strip()
                if key in lower_map:
                    return lower_map[key]
            return None

        location_col = resolve_col(["location", "location id", "location_id"])
        cat_col = resolve_col(["cat", "category"])

        numeric_cols = detect_numeric_features(df_mat, exclude_cols=[location_col, cat_col])
        if not numeric_cols:
            raise ValueError("No numeric features detected in matrix source to build PCA.")

        # Prepare numeric matrix
        X_num = df_mat[numeric_cols].apply(pd.to_numeric, errors="coerce").values

        # Impute and scale
        imputer = SimpleImputer(strategy="mean")
        X_imputed = imputer.fit_transform(X_num)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_imputed)

        # PCA
        n_components = min(X_scaled.shape[0], X_scaled.shape[1])
        pca = PCA(n_components=n_components)
        scores = pca.fit_transform(X_scaled)

        # Loadings: features x PCs (components_.T)
        pc_cols = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
        loadings = pd.DataFrame(pca.components_.T, index=numeric_cols, columns=pc_cols)

        # Save loadings (pca_loadings.csv) and also save rotation alias and scores/explained for completeness
        loadings.to_csv(LOADINGS_FILE)
        loadings.to_csv(ROTATION_FILE)  # alias for other scripts
        scores_df = pd.DataFrame(scores, columns=pc_cols, index=df_mat.index)
        # insert optional location/cat if present
        if location_col:
            scores_df.insert(0, location_col, df_mat[location_col].astype(str).values)
        else:
            scores_df.insert(0, "Location", [""] * len(scores_df))
        if cat_col:
            scores_df.insert(1, cat_col, df_mat[cat_col].astype(str).values)
        else:
            scores_df.insert(1, "Cat", [""] * len(scores_df))
        scores_df.to_csv(SCORES_FILE, index=False)
        explained_df = pd.DataFrame({
            "PC": pc_cols,
            "percent_explained": 100.0 * pca.explained_variance_ratio_
        })
        explained_df.to_csv(EXPLAINED_FILE, index=False)

        print(f"Built PCA and wrote loadings to {LOADINGS_FILE}, rotation alias {ROTATION_FILE}, scores to {SCORES_FILE}, explained to {EXPLAINED_FILE}.")

# ---- Load the loadings file for plotting ----
df = pd.read_csv(LOADINGS_FILE, index_col=0)
if df.empty:
    raise ValueError("pca_loadings.csv appears empty after attempted recovery/build.")

# Resolve PC names (case-insensitive), ensure PC3 exists
resolved = resolve_pc_names_from_df(df, PC_NAMES_PREFERRED)
if resolved[0] is None or resolved[1] is None:
    raise KeyError(f"Could not find PC1/PC2 columns in loadings. Columns found: {list(df.columns)}")
if resolved[2] is None:
    df["PC3"] = 0.0
    resolved[2] = "PC3"

pc1_col, pc2_col, pc3_col = resolved

# Ensure numeric
for c in (pc1_col, pc2_col, pc3_col):
    df[c] = pd.to_numeric(df[c], errors="coerce")

# List of all variables
variables = list(df.index)
n_vars = len(variables)
print(f"Found {n_vars} variables. Will plot all of them (from max to min for each PC).")

# Build consistent color map for variables (deterministic)
cmap = mpl.cm.get_cmap("tab20", max(20, n_vars))
colors = [mpl.colors.rgb2hex(cmap(i % cmap.N)) for i in range(n_vars)]
color_map = {var: colors[i] for i, var in enumerate(sorted(variables))}

# Prepare per-PC sorted series (max -> min)
s_pc1 = df[pc1_col].sort_values(ascending=False)
s_pc2 = df[pc2_col].sort_values(ascending=False)
s_pc3 = df[pc3_col].sort_values(ascending=False)

# Save ordered CSV with rank columns for each PC
ordered_df = pd.DataFrame(index=sorted(variables))
ordered_df[pc1_col] = df.loc[ordered_df.index, pc1_col]
ordered_df[pc2_col] = df.loc[ordered_df.index, pc2_col]
ordered_df[pc3_col] = df.loc[ordered_df.index, pc3_col]
ordered_df[f"{pc1_col}_rank_desc"] = ordered_df[pc1_col].rank(method="dense", ascending=False).astype(int)
ordered_df[f"{pc2_col}_rank_desc"] = ordered_df[pc2_col].rank(method="dense", ascending=False).astype(int)
ordered_df[f"{pc3_col}_rank_desc"] = ordered_df[pc3_col].rank(method="dense", ascending=False).astype(int)
ordered_df.to_csv(OUT_CSV)
print(f"Wrote ordered loadings CSV to: {OUT_CSV}")

# Plotting: three horizontal subplots with variable names on each vertical axis
plt.rcParams.update({"font.size": FONT_SIZE})
fig, axes = plt.subplots(ncols=3, nrows=1, figsize=FIGSIZE, constrained_layout=False)

pc_series = [(pc1_col, s_pc1), (pc2_col, s_pc2), (pc3_col, s_pc3)]
for ax, (pcname, series) in zip(axes, pc_series):
    vars_order = list(series.index)
    y_pos = np.arange(len(vars_order))
    values = series.values
    bar_colors = [color_map[v] for v in vars_order]

    ax.barh(y_pos, values, color=bar_colors, edgecolor="k", linewidth=0.25)
    ax.set_yticks(y_pos)
    # Show y-labels on every subplot for fast reading
    ax.set_yticklabels(vars_order, fontsize=FONT_SIZE - 1)
    ax.invert_yaxis()
    ax.set_xlabel(f"{pcname} loading", fontsize=FONT_SIZE + 1)
    ax.axvline(0, color="0.35", linewidth=0.9)
    ax.grid(axis="x", linestyle="--", linewidth=0.4, alpha=0.6)

    # Annotate numeric values (small but readable)
    vmax = np.nanmax(np.abs(values)) if values.size else 0.0
    offset = 0.005 * max(1.0, vmax)
    for yi, val in enumerate(values):
        ha = "left" if val >= 0 else "right"
        xpos = val + (offset if val >= 0 else -offset)
        ax.text(xpos, yi, f"{val:.3f}", va="center", ha=ha, fontsize=FONT_SIZE - 2, color="black")

# Remove legend entirely (user requested)
plt.subplots_adjust(left=0.06, right=0.98, wspace=0.18, top=0.94, bottom=0.04)
fig.suptitle("PCA loadings: all variables (max -> min per PC). Variable names on every vertical axis.", fontsize=FONT_SIZE + 2)

# Save high-resolution PNG and PDF for screenshots/doc inclusion
t1 = time.time()
fig.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
fig.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.close(fig)
t2 = time.time()

print(f"Wrote PNG to {OUT_PNG} and PDF to {OUT_PDF} (save time {t2-t1:.2f}s). Total runtime {time.time()-t0:.2f}s.")

# Cleanup
try:
    del df, ordered_df, loadings
except Exception:
    pass
gc.collect()
sys.exit(0)

Recovered loadings from C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_rotation.csv -> wrote C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_loadings.csv.
Found 22 variables. Will plot all of them (from max to min for each PC).
Wrote ordered loadings CSV to: C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_loadings_all_ordered.csv


C:\Users\kylian.robinson\AppData\Local\Temp\ipykernel_10996\3210248554.py:188: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.



Wrote PNG to C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_loadings_all_pcs_labeled_big.png and PDF to C:\Users\kylian.robinson\MapRebuildItems\V 2.1 items\results_3.1\pca_loadings_all_pcs_labeled_big.pdf (save time 1.75s). Total runtime 1.85s.


SystemExit: 0

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning:

To exit: use 'exit', 'quit', or Ctrl-D.

